In [ ]:
#This is for installing Pytorch, Ultralytics, Supervision, and other dependencies. Run this cell first before running the rest of the notebook and run this one only "once" for the first time. You may delete it next.
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
%pip install ultralytics supervision opencv-python ipywidgets matplotlib Pillow numpy

%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
%pip install ultralytics supervision opencv-python ipywidgets matplotlib Pillow numpy

# 1. Remove the headless build that was shadowing the GUI one
!pip uninstall -y opencv-python-headless

# 2. Force a clean reinstall of the GUI build (headless shared the cv2/ folder)
!pip install --force-reinstall --no-cache-dir opencv-python==4.13.0.92


In [ ]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
# Cell 2: Imports & Configuration
import cv2
import torch
import supervision as sv
from ultralytics import YOLO
from IPython.display import display, clear_output
# ipywidgets is not required in this cell
import numpy as np

# Paths
MODEL_PATH = r"/kaggle/input/datasets/towfiqurrashid/mohakhali/best_map50_0.73 - Copy.pt"

# ── BATCH MODE: put your 12 video paths here (one run → one Excel per video) ──
VIDEO_PATHS = [
    r"/kaggle/input/datasets/towfiqurrashid/mohakhali/1630-1645--ch042_20251224003247.mp4",
    r"/kaggle/input/datasets/towfiqurrashid/mohakhali/1730-1745--ch042_20251224000856.mp4",
    r"/kaggle/input/datasets/towfiqurrashid/mohakhali/ch042_20251223142121.mp4",
    r"/kaggle/input/datasets/towfiqurrashid/mohakhali/ch042_20251223142600.mp4",
    r"/kaggle/input/datasets/towfiqurrashid/mohakhali/ch042_20251223143328.mp4",
    r"/kaggle/input/datasets/towfiqurrashid/mohakhali/ch042_20251223143746.mp4",
    r"/kaggle/input/datasets/towfiqurrashid/mohakhali/ch042_20251223144136.mp4",
    r"/kaggle/input/datasets/towfiqurrashid/mohakhali/ch042_20251223144649.mp4",
    r"/kaggle/input/datasets/towfiqurrashid/mohakhali/ch042_20251223145123.mp4",
    r"/kaggle/input/datasets/towfiqurrashid/mohakhali/ch042_20251223145338.mp4",
    r"/kaggle/input/datasets/towfiqurrashid/mohakhali/ch042_20251223145645.mp4",
    r"/kaggle/input/datasets/towfiqurrashid/mohakhali/ch042_20251224000229.mp4"
]

# All Excel reports / annotated videos will be written here:
OUTPUT_DIR = r"/kaggle/working/excel reports"

# VIDEO_PATH (single) is kept ONLY for the utility cells
# (polygon point selector / frame preview). The batch loop ignores it.
VIDEO_PATH = VIDEO_PATHS[0]

# Class names
CLASS_NAMES = {
0: 'rickshaw', 1: 'car', 2: 'bike', 3: 'bus', 4: 'cng', 5: 'bicycle', 6: 'truck', 7: 'van', 8: 'leguna'
}

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3: Load Model
model = YOLO(MODEL_PATH)
model.to(device)

# Compatibility patch: older checkpoints pickled AAttn before `all_head_dim` was an attribute
from ultralytics.nn.modules.block import AAttn
for _m in model.model.modules():
    if isinstance(_m, AAttn) and not hasattr(_m, 'all_head_dim'):
        _m.all_head_dim = _m.head_dim * _m.num_heads

print("✅ Model loaded successfully!")
print(f"Model classes: {model.names}")

In [ ]:
# Cell 4: Define Annotator & Color Palette
color = sv.ColorPalette.from_hex([
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00",
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#ff3333", "#33cc33", "#ff6600"
])

box_annotator = sv.BoxAnnotator(color=color)
label_annotator = sv.LabelAnnotator(
    color=color,
    text_color=sv.Color.BLACK,
    smart_position=True
)

def annotate_frame(frame: np.ndarray, detections: sv.Detections) -> np.ndarray:
    """Annotate a single video frame with detections."""
    def _disp_name(class_id):
        return CLASS_NAMES.get(class_id, 'unknown')

    labels = [
        f"{_disp_name(class_id)} {confidence:.2f}"
        for class_id, confidence in zip(detections.class_id, detections.confidence)
    ] if len(detections) > 0 else []

    annotated = box_annotator.annotate(frame.copy(), detections)
    annotated = label_annotator.annotate(annotated, detections, labels=labels)
    return annotated

!pip install jupyterlab_widgets

import sys
!{sys.executable} -m pip install ipywidgets

# Cell 6: To verify if video output was saved
import os

# Safely get frame_count in case it was never initialized
frame_count  = frame_count  if 'frame_count'  in dir() else 0
total_frames = total_frames if 'total_frames' in dir() else '?'

if os.path.exists(OUTPUT_PATH):
    size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
    print(f"✅ Output file exists: {OUTPUT_PATH}")
    print(f"📦 File size: {size_mb:.2f} MB")
    print(f"🎞  Frames processed before stop: {frame_count}/{total_frames}")
else:
    print("❌ Output file not found — the writer may not have flushed.")
    print("👉 Re-run Cell 5 and let it complete fully without interrupting.")

# Cell: Interactive Polygon Point Selector
# Run this cell to click on the video frame and get polygon_points
# for use in Cell 11.
#
# Controls:
#   Left Click  → add a point
#   Right Click → undo last point
#   C           → clear all points
#   Enter       → confirm polygon and print coordinates
#   Q           → quit without saving

import cv2
import numpy as np

# ── Load first frame ──
cap = cv2.VideoCapture(VIDEO_PATH)
ret, first_frame = cap.read()
cap.release()

if not ret:
    raise RuntimeError("Cannot read first frame from video.")

orig_h, orig_w = first_frame.shape[:2]
print(f"Video resolution: {orig_w} x {orig_h}")

# ── Scale to fit screen ──
SCREEN_W = 1366
scale     = SCREEN_W / orig_w
screen_h  = int(orig_h * scale)
display_frame = cv2.resize(first_frame, (SCREEN_W, screen_h))

print(f"Display size   : {SCREEN_W} x {screen_h}  (scale={scale:.4f})")
print("\nLeft Click = add point | Right Click = undo | C = clear | Enter = confirm | Q = quit\n")

polygon_display = []   # click coords in display space (for drawing)
polygon_orig    = []   # click coords mapped back to original resolution

WINDOW = "Polygon Selector  |  Enter=confirm  Q=quit"

def redraw():
    img = display_frame.copy()
    for idx, pt in enumerate(polygon_display):
        cv2.circle(img, pt, 6, (0, 255, 0), -1)
        cv2.circle(img, pt, 8, (255, 255, 255), 1)
        cv2.putText(img, str(idx + 1), (pt[0] + 10, pt[1] - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)
        if idx > 0:
            cv2.line(img, polygon_display[idx - 1], pt, (0, 255, 0), 2)
    if len(polygon_display) > 2:
        cv2.line(img, polygon_display[-1], polygon_display[0], (0, 200, 0), 1)
    cv2.imshow(WINDOW, img)

def on_mouse(event, x, y, flags, param):
    ox = int(x / scale)
    oy = int(y / scale)
    if event == cv2.EVENT_LBUTTONDOWN:
        polygon_display.append((x, y))
        polygon_orig.append((ox, oy))
        print(f"  Point {len(polygon_orig):>2}: original=({ox}, {oy})")
        redraw()
    elif event == cv2.EVENT_RBUTTONDOWN and polygon_orig:
        removed = polygon_orig.pop()
        polygon_display.pop()
        print(f"  Removed: {removed}  |  {len(polygon_orig)} points remaining")
        redraw()

cv2.namedWindow(WINDOW, cv2.WINDOW_AUTOSIZE)
cv2.setMouseCallback(WINDOW, on_mouse)
cv2.imshow(WINDOW, display_frame)

confirmed = False
while True:
    key = cv2.waitKey(20) & 0xFF

    if key == 13:  # Enter — confirm
        if len(polygon_orig) >= 3:
            # Draw filled confirmation overlay
            img = display_frame.copy()
            pts = np.array(polygon_display, np.int32)
            overlay = img.copy()
            cv2.fillPoly(overlay, [pts], (0, 255, 0))
            cv2.addWeighted(overlay, 0.25, img, 0.75, 0, img)
            cv2.polylines(img, [pts], True, (0, 255, 0), 2)
            cv2.imshow(WINDOW, img)
            confirmed = True
            print("\nPolygon confirmed!")
            break
        else:
            print("Need at least 3 points.")

    elif key in (ord('c'), ord('C')):
        polygon_display.clear()
        polygon_orig.clear()
        redraw()
        print("  Cleared.")

    elif key in (ord('q'), ord('Q')):
        print("Quit — no polygon saved.")
        break

cv2.destroyAllWindows()

# ── Print coordinates ready to paste into Cell 11 ──
if confirmed and polygon_orig:
    print("\n" + "=" * 55)
    print(f"  {len(polygon_orig)} points at original resolution ({orig_w}x{orig_h})")
    print("=" * 55)
    print(f"\npolygon_points = {polygon_orig}")
    print(f"\nCopy the line above and paste it into Cell 11.")
    print("=" * 55)

%pip install openpyxl

results = model.predict(
    source=frame, conf=0.10, iou=0.45, imgsz=1280,
    device=device, verbose=False, half=True,
    save=False, save_txt=False, save_crop=False,   # don't write outputs
    project=r"C:\Users\MSI\yolo_runs",              # absolute path on a good drive
    name="predict", exist_ok=True,                  # avoid increment_path probing
)[0]


import os
from pathlib import Path
print("CWD:", os.getcwd())
print(Path("runs/detect/predict").resolve())
try:
    Path("runs/detect/predict").exists()
    print("exists() OK")
except OSError as e:
    print("FAILED:", e)


In [ ]:
import cv2
import numpy as np
import math
import os
from datetime import datetime
import supervision as sv
from supervision import ByteTrack  # Use ByteTrack directly from supervision
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.chart import BarChart, Reference
import time
from IPython.display import display, clear_output, Image as IPImage

# ══════════════════════════════════════════════════
# ── CONFIGURATION ──
# ══════════════════════════════════════════════════

# ───────────────────────────────────────────────────────────────────────────
#  SPEED LOG-ARRIVAL : the EXISTING two speed lines (renamed).
#  A vehicle's speed = ARRIVAL_DISTANCE_METERS / (time between crossing
#  Arrival-Line-1 and Arrival-Line-2).
# ───────────────────────────────────────────────────────────────────────────
ARR_LINE1_START = (727, 121)
ARR_LINE1_END   = (1437, 1175)
ARR_LINE2_START = (852, 127)
ARR_LINE2_END   = (1514, 1094)
ARRIVAL_DISTANCE_METERS = 1.42       # real-world gap between the two ARRIVAL lines

# ───────────────────────────────────────────────────────────────────────────
#  SPEED LOG-SERVICE : the NEW line + the FORMER COUNTING line (renamed).
#  Works with EXACTLY the same two-line logic as the arrival pair.
#  ⚠ SET SRV_LINE1 with the polygon point selector — the coordinates below
#    are PLACEHOLDERS (the old counting line shifted by 50 px).
#  ⚠ HARD-CODE the real-world gap between the two SERVICE lines below.
# ───────────────────────────────────────────────────────────────────────────
SRV_LINE1_START = (1641, 301)         # ← NEW line  (PLACEHOLDER — set me!)
SRV_LINE1_END   = (1904, 614)         # ← NEW line  (PLACEHOLDER — set me!)
SRV_LINE2_START = (1735, 316)         # ← former COUNTING line
SRV_LINE2_END   = (1990, 597)         # ← former COUNTING line
SERVICE_DISTANCE_METERS = 0.94       # ← real-world gap between the two SERVICE lines (HARD-CODE)

# --- Display / preview ---
DISPLAY_WIDTH, DISPLAY_HEIGHT = 800, 450
PREVIEW_EVERY = 10

# --- Outputs: computed PER VIDEO inside process_video() below ---
try:
    OUTPUT_DIR
except NameError:
    OUTPUT_DIR = r"D:\Thesis Mohakhali\excel reports"   # fallback if Cell 2 was not run
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = {
    0: 'rickshaw', 1: 'car', 2: 'bike', 3: 'bus', 4: 'cng',
    5: 'bicycle', 6: 'truck', 7: 'van', 8: 'leguna'
}

# ══════════════════════════════════════════════════
# ── HELPER FUNCTIONS ──
# ══════════════════════════════════════════════════
def side_of_line(pt, p1, p2):
    """Cross-product sign: >0 one side, <0 the other."""
    return (p2[0]-p1[0])*(pt[1]-p1[1]) - (p2[1]-p1[1])*(pt[0]-p1[0])

def get_bottom_center(xyxy):
    x1, y1, x2, y2 = xyxy
    return (int((x1 + x2) / 2), int(y2))

def fmt_ts(sec):
    return f"{int(sec//60):02d}:{sec%60:05.2f}"

# ══════════════════════════════════════════════════
# ── TWO-LINE SPEED-PAIR ENGINE ──
#    One reusable implementation of the existing Speed-Log logic, used for
#    BOTH pairs:  ARRIVAL (old speed lines)  and  SERVICE (new line + old
#    counting line). Each pair produces its own Speed-Log sheet.
# ══════════════════════════════════════════════════
def make_pair_state():
    return {
        'sides_L1': {},          # track_id → last side of line 1
        'sides_L2': {},          # track_id → last side of line 2
        'first':    {},          # track_id → (frame, line_name, enter_ts)
        'speeds':   {},          # track_id → last measured speed (km/h)
        'counted':  set(),       # track_ids already logged (each vehicle once)
        'counts':   {name: 0 for name in CLASS_NAMES.values()},
        'log':      [],          # [enter_frame, enter_time, exit_frame, exit_time, id, class, conf, speed]
    }

def update_pair(st, L1S, L1E, L2S, L2E, dist_m,
                track_id, class_name, confidence, pt, frame_count, fps):
    """Same crossing/speed logic as the original Speed Log, parameterised."""
    cur1 = side_of_line(pt, L1S, L1E)
    crossed1 = False
    if track_id in st['sides_L1']:
        prev = st['sides_L1'][track_id]
        if (prev > 0 and cur1 <= 0) or (prev <= 0 and cur1 > 0):
            crossed1 = True
    st['sides_L1'][track_id] = cur1

    cur2 = side_of_line(pt, L2S, L2E)
    crossed2 = False
    if track_id in st['sides_L2']:
        prev = st['sides_L2'][track_id]
        if (prev > 0 and cur2 <= 0) or (prev <= 0 and cur2 > 0):
            crossed2 = True
    st['sides_L2'][track_id] = cur2

    for crossed, line_name in [(crossed1, "L1"), (crossed2, "L2")]:
        if not crossed:
            continue
        if track_id not in st['first']:
            st['first'][track_id] = (frame_count, line_name, frame_count / fps)
        else:
            first_frame, first_line, enter_ts = st['first'][track_id]
            if line_name != first_line:
                dt_frames = frame_count - first_frame
                speed_kmh = None
                if dt_frames > 0:
                    t         = dt_frames / fps
                    speed_kmh = (dist_m / t) * 3.6
                    st['speeds'][track_id] = speed_kmh
                if track_id not in st['counted']:
                    st['counts'][class_name] = st['counts'].get(class_name, 0) + 1
                    st['counted'].add(track_id)
                    exit_ts = frame_count / fps
                    st['log'].append([
                        first_frame, fmt_ts(enter_ts), frame_count, fmt_ts(exit_ts),
                        track_id, class_name, round(confidence, 3),
                        round(speed_kmh, 1) if speed_kmh is not None else "N/A"
                    ])
                del st['first'][track_id]

# ══════════════════════════════════════════════════
# ── PER-VIDEO PIPELINE ──
#    Everything below (tracker state → Excel writer → inference loop)
#    lives inside process_video(), so each video starts with a FRESH
#    tracker and fresh counters, and writes its OWN Excel file.
#    Run the BATCH DRIVER cell (last cell) to process all videos.
# ══════════════════════════════════════════════════
def process_video(VIDEO_PATH):
    """Run detection + tracking on one video. Returns (EXCEL_OUTPUT, VIDEO_OUTPUT)."""
    # --- per-video output paths ---
    video_basename = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
    EXCEL_OUTPUT   = os.path.join(OUTPUT_DIR, f"{video_basename}_report.xlsx")
    VIDEO_OUTPUT   = os.path.join(OUTPUT_DIR, f"{video_basename}_annotated.mp4")

    # ══════════════════════════════════════════════════
    # ── TRACKING STATE (one shared tracker, TWO speed pairs) ──
    # ══════════════════════════════════════════════════
    tracker       = ByteTrack()
    track_classes = {}                  # track_id → class_name
    arrival       = make_pair_state()   # Speed Log-Arrival  (old speed lines)
    service       = make_pair_state()   # Speed Log-Service  (new line + old counting line)

    # ══════════════════════════════════════════════════
    # ── EXCEL: Summary + Speed Log-Arrival + Speed Log-Service ──
    #    (the Per-N-Seconds and Count-Log sheets are REMOVED — the former
    #     counting line is now the 2nd line of the SERVICE pair)
    # ══════════════════════════════════════════════════
    def save_excel(frame_count, total_frames, fps, status="complete"):
        wb = openpyxl.Workbook()
        HDR_BG, TOTAL_BG = "1F3864", "2E75B6"
        thin = Border(left=Side(style='thin'), right=Side(style='thin'),
                      top=Side(style='thin'), bottom=Side(style='thin'))

        def style_header(cell, bg=HDR_BG):
            cell.font = Font(bold=True, color="FFFFFF", name='Arial', size=11)
            cell.fill = PatternFill("solid", start_color=bg)
            cell.alignment = Alignment(horizontal='center', vertical='center')
            cell.border = thin

        def style_data(cell, alt=False):
            cell.font = Font(name='Arial', size=10)
            cell.fill = PatternFill("solid", start_color="DCE6F1" if alt else "FFFFFF")
            cell.alignment = Alignment(horizontal='center', vertical='center')
            cell.border = thin

        status_str = "✅ Complete" if status == "complete" else f"⚠️ Interrupted @ frame {frame_count}/{total_frames}"

        # ───────────── Sheet 1: Summary (class counts from the SERVICE pair) ─────────────
        #  All downstream analysis volumes are taken from Speed Log-SERVICE,
        #  so the Summary counts come from the SERVICE pair too.
        ws1 = wb.active
        ws1.title = "Summary"
        ws1.merge_cells("A1:C1")
        ws1["A1"] = "🚦 Vehicle Count + Speed Report"
        ws1["A1"].font = Font(bold=True, size=14, color="1F3864", name='Arial')
        ws1["A1"].alignment = Alignment(horizontal='center')

        meta = [
            ("Video",                 os.path.basename(VIDEO_PATH)),
            ("Arrival Line 1",        f"{ARR_LINE1_START} → {ARR_LINE1_END}"),
            ("Arrival Line 2",        f"{ARR_LINE2_START} → {ARR_LINE2_END}"),
            ("Arrival distance (m)",  ARRIVAL_DISTANCE_METERS),
            ("Service Line 1 (new)",  f"{SRV_LINE1_START} → {SRV_LINE1_END}"),
            ("Service Line 2 (old counting line)", f"{SRV_LINE2_START} → {SRV_LINE2_END}"),
            ("Service distance (m)",  SERVICE_DISTANCE_METERS),
            ("Generated",             datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
            ("Status",                status_str),
            ("Frames Done",           f"{frame_count} / {total_frames}"),
            ("FPS",                   round(fps, 2)),
        ]
        for i, (k, v) in enumerate(meta, 2):
            ws1.cell(i, 1).value = k
            ws1.cell(i, 2).value = v
            ws1.cell(i, 1).font = Font(bold=True, name='Arial', size=10)
            ws1.cell(i, 2).font = Font(name='Arial', size=10)

        hdr_row = len(meta) + 3
        for col, h in enumerate(["Class", "Count", "% of Total"], 1):
            style_header(ws1.cell(hdr_row, col))
            ws1.cell(hdr_row, col).value = h

        sorted_counts = sorted(service['counts'].items(), key=lambda x: x[1], reverse=True)
        data_start = hdr_row + 1
        for i, (cls, cnt) in enumerate(sorted_counts):
            row = data_start + i
            ws1.cell(row, 1).value = cls
            ws1.cell(row, 2).value = cnt
            ws1.cell(row, 3).value = f"=IF(B{data_start+len(sorted_counts)}=0,0,B{row}/B{data_start+len(sorted_counts)}*100)"
            ws1.cell(row, 3).number_format = '0.00"%"'
            for col in range(1, 4):
                style_data(ws1.cell(row, col), alt=(i % 2 == 1))

        total_row = data_start + len(sorted_counts)
        ws1.cell(total_row, 1).value = "TOTAL"
        ws1.cell(total_row, 2).value = f"=SUM(B{data_start}:B{total_row-1})"
        ws1.cell(total_row, 3).value = "100.00%"
        for col in range(1, 4):
            c = ws1.cell(total_row, col)
            c.font = Font(bold=True, color="FFFFFF", name='Arial', size=11)
            c.fill = PatternFill("solid", start_color=TOTAL_BG)
            c.alignment = Alignment(horizontal='center')
            c.border = thin

        ws1.column_dimensions['A'].width = 30
        ws1.column_dimensions['B'].width = 26
        ws1.column_dimensions['C'].width = 16

        chart = BarChart()
        chart.type = "col"; chart.title = "Vehicle Counts by Class (Speed Log-Service)"
        chart.y_axis.title = "Count"; chart.x_axis.title = "Class"
        chart.style = 10; chart.width = 20; chart.height = 14
        chart.add_data(Reference(ws1, min_col=2, min_row=hdr_row, max_row=total_row-1), titles_from_data=True)
        chart.set_categories(Reference(ws1, min_col=1, min_row=data_start, max_row=total_row-1))
        ws1.add_chart(chart, "E8")

        # ───────────── Sheets 2 & 3: the two Speed Logs ─────────────
        def write_speed_log(sheet_name, rows):
            ws = wb.create_sheet(sheet_name)
            for col, h in enumerate(["Enter Frame", "Enter Time", "Exit Frame", "Exit Time",
                                     "Track ID", "Class", "Confidence", "Speed (km/h)"], 1):
                style_header(ws.cell(1, col))
                ws.cell(1, col).value = h
            for i, row_data in enumerate(rows):
                for col, val in enumerate(row_data, 1):
                    ws.cell(i+2, col).value = val
                    style_data(ws.cell(i+2, col), alt=(i % 2 == 1))
            for col, w in enumerate([12, 14, 12, 14, 12, 14, 14, 16], 1):
                ws.column_dimensions[get_column_letter(col)].width = w

        write_speed_log("Speed Log-Arrival", arrival['log'])   # old two speed lines
        write_speed_log("Speed Log-Service", service['log'])   # new line + old counting line

        wb.save(EXCEL_OUTPUT)
        print(f"\n📊 Combined report saved → {EXCEL_OUTPUT}")
        print(f"   Frames processed        : {frame_count}/{total_frames}")
        print(f"   Speed Log-Arrival count : {sum(arrival['counts'].values())}")
        print(f"   Speed Log-Service count : {sum(service['counts'].values())}")

    # ══════════════════════════════════════════════════
    # ── VIDEO INFERENCE LOOP (single pass) ──
    # ══════════════════════════════════════════════════
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise FileNotFoundError(f"❌ Cannot open: {VIDEO_PATH}")

    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps          = cap.get(cv2.CAP_PROP_FPS) or 25
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    OUT_W, OUT_H = width // 2, height // 2
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(VIDEO_OUTPUT, fourcc, fps, (OUT_W, OUT_H))

    print(f"📹 {width}x{height} @ {fps:.1f} FPS | {total_frames} frames")
    print(f"📹 Output video: {OUT_W}x{OUT_H} (half resolution)")
    print(f"📏 ARRIVAL Line1 : {ARR_LINE1_START} → {ARR_LINE1_END}")
    print(f"📏 ARRIVAL Line2 : {ARR_LINE2_START} → {ARR_LINE2_END}")
    print(f"📐 ARRIVAL dist  : {ARRIVAL_DISTANCE_METERS} m")
    print(f"📏 SERVICE Line1 : {SRV_LINE1_START} → {SRV_LINE1_END}   (NEW line)")
    print(f"📏 SERVICE Line2 : {SRV_LINE2_START} → {SRV_LINE2_END}   (old counting line)")
    print(f"📐 SERVICE dist  : {SERVICE_DISTANCE_METERS} m\n")

    FONT = cv2.FONT_HERSHEY_SIMPLEX
    frame_count = 0
    _last_preview = 0.0

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            results    = model.predict(source=frame, conf=0.10, iou=0.45, imgsz=1280,
                                        device=device, verbose=False, half=True)[0]
            detections = sv.Detections.from_ultralytics(results)
            detections = tracker.update_with_detections(detections)

            n = len(detections) if detections.tracker_id is not None else 0
            for i in range(n):
                if detections.tracker_id[i] is None:
                    continue
                track_id   = int(detections.tracker_id[i])
                class_id   = int(detections.class_id[i])
                confidence = float(detections.confidence[i])
                class_name = CLASS_NAMES.get(class_id, 'unknown')
                cx, cy     = get_bottom_center(detections.xyxy[i])
                track_classes[track_id] = class_name

                # ───────── A) SPEED LOG-ARRIVAL (old two speed lines) ─────────
                update_pair(arrival,
                            ARR_LINE1_START, ARR_LINE1_END, ARR_LINE2_START, ARR_LINE2_END,
                            ARRIVAL_DISTANCE_METERS,
                            track_id, class_name, confidence, (cx, cy), frame_count, fps)

                # ───────── B) SPEED LOG-SERVICE (new line + old counting line) ─────────
                update_pair(service,
                            SRV_LINE1_START, SRV_LINE1_END, SRV_LINE2_START, SRV_LINE2_END,
                            SERVICE_DISTANCE_METERS,
                            track_id, class_name, confidence, (cx, cy), frame_count, fps)

            # ───────── ANNOTATE ─────────
            annotated = annotate_frame(frame, detections)

            # ARRIVAL pair (blue + red) with yellow endpoints
            cv2.line(annotated, ARR_LINE1_START, ARR_LINE1_END, (255, 0, 0), 3)
            cv2.line(annotated, ARR_LINE2_START, ARR_LINE2_END, (0, 0, 255), 3)
            for p in (ARR_LINE1_START, ARR_LINE1_END, ARR_LINE2_START, ARR_LINE2_END):
                cv2.circle(annotated, p, 8, (0, 255, 255), -1)
            cv2.putText(annotated, "ARRIVAL",
                        (ARR_LINE1_START[0], ARR_LINE1_START[1] + 30), FONT, 0.8, (0, 255, 255), 2)

            # SERVICE pair (orange = NEW line, magenta = old counting line)
            cv2.line(annotated, SRV_LINE1_START, SRV_LINE1_END, (0, 165, 255), 3)
            cv2.line(annotated, SRV_LINE2_START, SRV_LINE2_END, (255, 0, 255), 3)
            for p in (SRV_LINE1_START, SRV_LINE1_END):
                cv2.circle(annotated, p, 8, (0, 165, 255), -1)
            for p in (SRV_LINE2_START, SRV_LINE2_END):
                cv2.circle(annotated, p, 8, (255, 0, 255), -1)
            cv2.putText(annotated, "SERVICE",
                        (SRV_LINE2_START[0], SRV_LINE2_START[1] - 12), FONT, 0.8, (255, 0, 255), 2)

            # speed labels on tracked vehicles (service speed preferred, else arrival)
            for i in range(n):
                if detections.tracker_id[i] is None:
                    continue
                tid = int(detections.tracker_id[i])
                spd = service['speeds'].get(tid, arrival['speeds'].get(tid))
                if spd is not None:
                    x1, y1, x2, y2 = detections.xyxy[i].astype(int)
                    cv2.putText(annotated, f"{spd:.0f} km/h", (x1, y1 - 10),
                                FONT, 0.7, (0, 255, 0), 2)

            total_arrival = sum(arrival['counts'].values())
            total_service = sum(service['counts'].values())

            # HUD: arrival & service pair counts (per-class from the SERVICE pair)
            cv2.putText(annotated, f"Arrival count: {total_arrival}", (30, 50), FONT, 1.1, (0, 255, 255), 3)
            cv2.putText(annotated, f"Service count: {total_service}", (30, 95), FONT, 1.1, (0, 255, 0), 3)
            y_off = 140
            for name, cnt in service['counts'].items():
                if cnt > 0:
                    cv2.putText(annotated, f"{name}: {cnt}", (30, y_off), FONT, 0.8, (255, 255, 0), 2)
                    y_off += 35

            _now = time.time()
            if frame_count % PREVIEW_EVERY == 0 or (_now - _last_preview >= 3.0):
                _last_preview = _now
                disp = cv2.cvtColor(cv2.resize(annotated, (DISPLAY_WIDTH, DISPLAY_HEIGHT)), cv2.COLOR_BGR2RGB)
                _, buf = cv2.imencode('.jpg', disp, [cv2.IMWRITE_JPEG_QUALITY, 75])
                clear_output(wait=True)
                display(IPImage(data=buf.tobytes()))
                print(f"⏳ Frame {frame_count}/{total_frames} ({frame_count/total_frames*100:.1f}%) | "
                      f"arrival: {total_arrival} | service: {total_service}")

            frame_count += 1

    except KeyboardInterrupt:
        print(f"\n⚠️  Interrupted at frame {frame_count}/{total_frames}")
        save_excel(frame_count, total_frames, fps, status="interrupted")
        raise

    else:
        save_excel(frame_count, total_frames, fps, status="complete")

    finally:
        cap.release()
        out.release()
        print(f"🎬 Video saved → {VIDEO_OUTPUT}")

    return EXCEL_OUTPUT, VIDEO_OUTPUT


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  Traffic Analysis & Delay-Cost Estimation
#  ----------------------------------------------------------------------------
#  Reads the Excel report produced by the cell above and writes ONE final
#  workbook that KEEPS every original sheet (Summary, Speed Log-Arrival,
#  Speed Log-Service) — data, formulas and charts intact — and APPENDS the
#  analysis sheets:
#     1 Arrival rate λ — from Speed Log-ARRIVAL
#       Service rate μ — from Speed Log-SERVICE (= count ÷ counting time)
#     2 Gate RED/GREEN — fixed hard-coded bin window on Speed Log-SERVICE
#     3 PCU                                   4 Delay (Hakkert-Gitelman)
#     5 Travel-time cost (hard-coded speed threshold; 0 if no RED period)
#     6 Fuel-loss cost                        7 Electricity cost
#     8 Pollution cost                        9 VOC
#    10 Cost Summary                         11 Whole-intersection estimate
#    12 Queue analysis (3 sheets): T_Q = μ·r/(μ−λ);  Q_M = λ·r/3600;
#       Q_L = Q_M × 20 / L_M   (λ, μ in LVU/h; r = effective RED, s)
#  All per-vehicle volumes, speeds, costs and the RED-period detection use
#  Speed Log-SERVICE; λ (arrival rate) uses Speed Log-ARRIVAL.
#  ONLY ONE Excel file is produced: "<video>_analysis.xlsx".  The intermediate
#  detection report is deleted after its sheets are merged into the final file.
#  Run AFTER the main detection cell (it reads that cell's EXCEL_OUTPUT).
# ════════════════════════════════════════════════════════════════════════════
import os, math
import numpy as np
from collections import Counter, OrderedDict
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ──────────────────────────────────────────────────────────────────────────
#  INPUT / OUTPUT PATHS
#  SRC_XLSX defaults to the workbook the detection cell just produced
#  (EXCEL_OUTPUT). If this cell is run on its own, it falls back to the
#  uploaded report file in the current directory.
# ──────────────────────────────────────────────────────────────────────────
# SRC_XLSX is now a PARAMETER of run_analysis() below — the batch driver
# passes the per-video report produced by process_video().

# Only ONE Excel file should remain on disk: the final "<video>_analysis.xlsx".
# After it is saved (with every original sheet preserved inside it), the
# intermediate detection report is deleted. Set to False to keep the report.
DELETE_SOURCE_REPORT = True

# ──────────────────────────────────────────────────────────────────────────
#  CONSTANTS & ASSUMPTIONS
# ──────────────────────────────────────────────────────────────────────────
FPS              = 25.0      # frames per second
ROAD_LENGTH_M    = 25.07      # metres — hard-coded road length used in the
                             # POLLUTION COST (change here to re-tune RL)
EC_IDLING_WH_MIN = 1.2       # Wh/min idling, battery rickshaw (Section 7)
P_ELEC_BDT_KWH   = 7.64      # BDT/kWh (Section 7)
VOC_WINDOW_MIN   = 15.0      # VOC is calculated for the 15-MINUTE video window

# Fuel prices (BDT / litre) — Section 6
P_DIESEL, P_PETROL, P_OCTANE, P_KEROSENE, P_CNG = 115.0, 140.0, 145.0, 135.0, 0.043

# Section 3 — PCU factors
PCU_FACTOR = {'rickshaw':2.0,'car':1.0,'cng':2.0,'truck':3.0,'bus':3.0,
              'leguna':1.5,'bike':0.75,'bicycle':0.5,'van':2.0}

# Section 5 — Value of Time (BDT / min, 2026)
VOT = {'car':12,'bus':6,'cng':5,'leguna':3,'bike':2,'rickshaw':1,
       'bicycle':1,'truck':2,'van':1}

# Section 6 — idling fuel use (ml/min) + fuel type/price per vehicle class
#   rickshaw = battery (no fuel, see Section 7);  bicycle & van = no fuel
FUEL_ML_MIN = {'car':11.75,'cng':24.50,'truck':16.70,'bus':15.5,'leguna':10.30,
               'bike':2.975,'rickshaw':0.0,'bicycle':0.0,'van':0.0}
FUEL_PRICE  = {'car':P_PETROL,'cng':P_CNG,'truck':P_DIESEL,'bus':P_DIESEL,
               'leguna':P_DIESEL,'bike':P_PETROL,'rickshaw':0.0,'bicycle':0.0,'van':0.0}
FUEL_LABEL  = {'car':'Petrol (Car)','cng':'LPG/CNG','truck':'Diesel (Truck)',
               'bus':'Diesel (Bus)','leguna':'Diesel (Tempo)','bike':'Petrol (Bike)',
               'rickshaw':'Electric (battery)','bicycle':'None','van':'None'}

# Section 8 — emission factors (g/km) by pollutant and EF-table category
EF = {
 'CO2':  {'bus':515.2,'micro':223.6,'tempo':208.3,'car':223.6,'cng':208.3,'bike':26.6,'lorry':887,'truck':887,'pickup':332.5},
 'CO':   {'bus':3.6,'micro':1.98,'tempo':0.9,'car':1.98,'cng':0.9,'bike':2.2,'lorry':0.68,'truck':0.68,'pickup':0.59},
 'NOx':  {'bus':12,'micro':0.2,'tempo':0.5,'car':0.2,'cng':0.5,'bike':0.19,'lorry':2.71,'truck':2.71,'pickup':2.37},
 'CH4':  {'bus':0.09,'micro':0.2,'tempo':0.14,'car':0.2,'cng':1.41,'bike':0.2,'lorry':0.06,'truck':0.06,'pickup':8.49},
 'SO2':  {'bus':1.4,'micro':0.05,'tempo':0.3,'car':0.05,'cng':0,'bike':0.013,'lorry':1,'truck':1,'pickup':0.3},
 'PM':   {'bus':0.6,'micro':0.07,'tempo':1.27,'car':0.03,'cng':0.13,'bike':0.05,'lorry':2.82,'truck':2.82,'pickup':0.14},
 'HC':   {'bus':0.9,'micro':0.25,'tempo':0.13,'car':0.21,'cng':0.255,'bike':1.42,'lorry':0.41,'truck':0.6,'pickup':0.05},
 'NMVOC':{'bus':0.21,'micro':1.27,'tempo':1.08,'car':0.08,'cng':1.09,'bike':0.13,'lorry':0.19,'truck':0.19,'pickup':0.17},
}
# Pollutant cost (BDT / kg) — Section 8
CP = {'PM':11215.50,'SO2':2493.00,'NOx':1528.50,'CH4':187.20,
      'NMVOC':25.50,'HC':25.50,'CO2':22.23,'CO':7.50}
POLLUTANTS = ['CO2','CO','NOx','CH4','SO2','PM','HC','NMVOC']
# vehicle class → EF-table category (rickshaw/bicycle are electric/none ⇒ no EF)
VEH_TO_EF = {'bus':'bus','car':'car','cng':'cng','truck':'truck',
             'leguna':'tempo','bike':'bike'}

# Section 9 — VEHICLE OPERATING COST  (Projected Financial VOC at IRI 14,
# FY 2025-26, Taka/km — values taken from the provided table)
#   truck  = avg. of Medium 83.2 + Small 62.7 = 73.0
VOC_TK_KM = {'truck':73.0, 'bus':99.0, 'car':77.1, 'leguna':24.5, 'cng':12.3,
             'bike':6.4, 'van':47.4, 'rickshaw':47.1, 'bicycle':47.1}
VOC_SOURCE_LABEL = {'truck':'Truck (avg. Medium 83.2 + Small 62.7)',
                    'bus':'Large Bus', 'car':'Car', 'leguna':'Leguna (Tempo)',
                    'cng':'CNG (Auto Rickshaw)', 'bike':'Motorcycle',
                    'van':'Van', 'rickshaw':'Rickshaw', 'bicycle':'Bicycle'}

# ── Section 2 — EFFECTIVE RED PERIOD: HARD-CODED fixed bin window ──
#    RED is detected by binning the Speed Log-SERVICE exit times into fixed
#    windows of RED_BIN_SECONDS; an empty bin (no arrival) confirmed by an empty
#    neighbour bin = RED. No Otsu, no auto bin-size search. ← EDIT the window here.
RED_BIN_SECONDS = 15.0   # fixed RED-detection window (s)  ← HARD-CODE HERE

# ── Section 5 — CONGESTION SPEED THRESHOLD: HARD-CODED (km/h) ──
#    A vehicle is "congested" if its Speed Log-SERVICE speed is BELOW this value.
#    No Otsu / percentile / median auto-detection. ← EDIT the threshold here.
CONG_SPEED_KMH = 5.0     # congested if speed < this many km/h  ← HARD-CODE HERE

# Section 12 — QUEUE ANALYSIS (formulas from the attached reference)
#   (1) T_Q = μ·r / (μ − λ)        Queue Duration Time (s)
#   (2) Q_M = λ·r / 3600           Maximum Queue Length (LVU)
#   (3) Q_L = Q_M × 20 / L_M       Vehicle Queue Length (m)
ROAD_ENTRANCE_WIDTH_M = 6.14   # L_M — width of road entrance (m). ← HARD-CODE HERE
QL_M_PER_LVU          = 20.0  # the constant '20' in formula (3)
# LVU (light-vehicle-unit) factors: counted vehicles → LVU for λ and μ in the
# queue formulas. Default 1.0 = plain vehicle counts. ← EDIT to your LVU table.
LVU_FACTOR = {'rickshaw':1.0,'car':1.0,'cng':1.0,'truck':1.0,'bus':1.0,
              'leguna':1.0,'bike':1.0,'bicycle':1.0,'van':1.0}

# ══════════════════════════════════════════════════════════════════════════
# ── PER-VIDEO ANALYSIS ──
#    The whole analysis is wrapped in run_analysis() so the batch driver can
#    call it once per video. One final "<video>_analysis.xlsx" per video.
# ══════════════════════════════════════════════════════════════════════════
def run_analysis(SRC_XLSX, VIDEO_PATH, VIDEO_OUTPUT=None):
    """Read one detection report and write <video>_analysis.xlsx. Returns its path."""
    # ──────────────────────────────────────────────────────────────────────────
    #  READ THE SOURCE WORKBOOK  (locate sheets by header signature, name-agnostic)
    # ──────────────────────────────────────────────────────────────────────────
    wb = openpyxl.load_workbook(SRC_XLSX)        # keeps original data + charts

    def _rows(ws, limit=None):
        out = []
        for i, r in enumerate(ws.iter_rows(values_only=True)):
            out.append(r)
            if limit and i + 1 >= limit:
                break
        return out

    def _find_header(ws, needed, scan=20):
        """Return (header_row_index, {colname: col_index}) for first row containing all `needed`."""
        for i, r in enumerate(_rows(ws, scan)):
            if not r:
                continue
            cells = [str(c).strip() if c is not None else '' for c in r]
            if all(any(n.lower() == c.lower() for c in cells) for n in needed):
                return i, {c: j for j, c in enumerate(cells)}
        return None, None

    def find_sheet(needed):
        """Find the worksheet whose header row contains all `needed` column names."""
        for ws in wb.worksheets:
            idx, cmap = _find_header(ws, needed)
            if idx is not None:
                return ws, idx, cmap
        raise KeyError(f"No sheet with columns {needed} found in {SRC_XLSX}")

    SPEED_HDR_COLS = ['Enter Frame', 'Exit Frame', 'Class', 'Speed (km/h)']

    def find_speed_sheet(role_keywords):
        """Prefer a Speed-Log sheet whose TITLE contains one of role_keywords."""
        for ws in wb.worksheets:
            if any(k in ws.title.lower() for k in role_keywords):
                idx, cmap = _find_header(ws, SPEED_HDR_COLS)
                if idx is not None:
                    return ws, idx, cmap
        return None, None, None

    ws_arr, arr_hdr, arr_cols = find_speed_sheet(['arrival'])
    ws_srv, srv_hdr, srv_cols = find_speed_sheet(['service'])
    if ws_srv is None:   # legacy workbook with a single 'Speed Log' → both roles
        ws_srv, srv_hdr, srv_cols = find_sheet(SPEED_HDR_COLS)
    if ws_arr is None:
        ws_arr, arr_hdr, arr_cols = ws_srv, srv_hdr, srv_cols
    print(f"📖 Source workbook : {SRC_XLSX}")
    print(f"   Arrival sheet : '{ws_arr.title}'   |   Service sheet : '{ws_srv.title}'")

    # ── Parse a Speed-Log sheet → traversal times, classes, speeds, exit frames ──
    def _parse_speed_sheet(ws, hdr, cols):
        c_ef, c_xf  = cols['Enter Frame'], cols['Exit Frame']
        c_cls, c_spd = cols['Class'], cols['Speed (km/h)']
        tt, cls_, spd, xf = [], [], [], []
        for r in _rows(ws)[hdr + 1:]:
            if not r or r[c_ef] is None or r[c_xf] is None:
                continue
            tt.append((float(r[c_xf]) - float(r[c_ef])) / FPS)
            xf.append(float(r[c_xf]))
            cls_.append(str(r[c_cls]).strip().lower())
            try:
                spd.append(float(r[c_spd]))
            except (TypeError, ValueError):
                spd.append(None)
        return tt, cls_, spd, xf

    # SERVICE pair → all per-vehicle volumes / speeds / traversal times (costs)
    traversal_times, veh_classes, veh_speeds, srv_exit_frames = \
        _parse_speed_sheet(ws_srv, srv_hdr, srv_cols)
    # ARRIVAL pair → λ and the RED-period detection
    _, arr_classes, _, arr_exit_frames = _parse_speed_sheet(ws_arr, arr_hdr, arr_cols)

    volume = Counter(veh_classes)         # V_i — per-class volume (Speed Log-SERVICE)
    veh_total = len(traversal_times)      # total vehicle count (Speed Log-SERVICE)
    arr_volume = Counter(arr_classes)     # per-class volume (Speed Log-ARRIVAL)
    arr_total  = len(arr_exit_frames)     # total vehicle count (Speed Log-ARRIVAL)

    # per-class AVERAGE SPEED (km/h) from Speed Log-SERVICE — used in the VOC (Section 9)
    _spd_sum, _spd_n = Counter(), Counter()
    for cl, v in zip(veh_classes, veh_speeds):
        if v is not None:
            _spd_sum[cl] += v
            _spd_n[cl]   += 1
    avg_speed = {cl: (_spd_sum[cl] / _spd_n[cl]) if _spd_n[cl] else 0.0 for cl in volume}

    # ════════════════════════════════════════════════════════════════════════════
    #  SECTION 1 — ARRIVAL RATE (λ) and SERVICE RATE (μ)
    #  λ = (vehicles in Speed Log-ARRIVAL) ÷ (observed time span) × 3600
    #  μ = (vehicles in Speed Log-SERVICE) ÷ (total counting time) × 3600
    #  Each observed time span is taken from its own sheet: last exit frame ÷ FPS.
    # ════════════════════════════════════════════════════════════════════════════
    obs_seconds = (max(arr_exit_frames) / FPS) if arr_exit_frames else 0.0
    lam_vph = (arr_total / obs_seconds) * 3600 if obs_seconds else 0.0    # λ veh/h
    q_vps   = lam_vph / 3600.0                                            # veh/s

    srv_obs_seconds = (max(srv_exit_frames) / FPS) if srv_exit_frames else 0.0
    mu_vph = (veh_total / srv_obs_seconds) * 3600 if srv_obs_seconds else 0.0  # μ veh/h
    mu_vps = mu_vph / 3600.0

    # LVU-converted rates for the queue formulas (Section 12)
    lam_lvu_h = (sum(arr_volume[c] * LVU_FACTOR.get(c, 1.0) for c in arr_volume)
                 / obs_seconds) * 3600 if obs_seconds else 0.0
    mu_lvu_h  = (sum(volume[c] * LVU_FACTOR.get(c, 1.0) for c in volume)
                 / srv_obs_seconds) * 3600 if srv_obs_seconds else 0.0

    # ════════════════════════════════════════════════════════════════════════════
    #  SECTION 2 — GATE OPENING / CLOSING TIMES
    #  RED detection from SPEED LOG-SERVICE counts, FIXED (HARD-CODED) bin window.
    #  Bin the service exit times into windows of RED_BIN_SECONDS:
    #    • an empty bin (no vehicle arrived) is a RED candidate;
    #    • it is CONFIRMED RED only if the previous OR next bin is also empty
    #      (neighbour confirmation — suppresses isolated random lulls).
    #  No Otsu, no auto bin-size search; the window is set by RED_BIN_SECONDS.
    # ════════════════════════════════════════════════════════════════════════════
    _exit_sec = np.sort(np.asarray(srv_exit_frames, dtype=float) / FPS)
    _obs      = float(_exit_sec.max()) if len(_exit_sec) else 0.0

    WIN_SEC    = float(RED_BIN_SECONDS)                 # hard-coded fixed window
    BIN_METHOD = f"fixed {WIN_SEC:g}s window (hard-coded)"

    n_win = max(int(np.ceil(_obs / WIN_SEC)), 1) if _obs > 0 else 0
    win_count = np.zeros(n_win, int)
    for t in _exit_sec:
        win_count[min(int(t // WIN_SEC), n_win - 1)] += 1

    def _is_red_candidate(i):
        return 0 <= i < n_win and win_count[i] == 0      # NO vehicle arrived

    def _derive_signal(i):
        return 'RED' if (_is_red_candidate(i) and
                         (_is_red_candidate(i - 1) or _is_red_candidate(i + 1))) else 'GREEN'

    signals = [_derive_signal(i) for i in range(n_win)]
    durs    = ([WIN_SEC] * (n_win - 1) + [_obs - (n_win - 1) * WIN_SEC]) if n_win else []

    print(f"RED bin size (hard-coded): {WIN_SEC:g} s")
    print(f"RED detection (Service)  : {n_win} bins, "
          f"{int((win_count == 0).sum()) if n_win else 0} empty, "
          f"{sum(1 for s in signals if s == 'RED')} confirmed RED bin(s)")
    # contiguous RED / GREEN runs  → distinct events
    runs, prev = [], None
    for s, d in zip(signals, durs):
        if s != prev:
            runs.append([s, d]); prev = s
        else:
            runs[-1][1] += d
    N_red   = sum(1 for s, _ in runs if s == 'RED')
    N_green = sum(1 for s, _ in runs if s == 'GREEN')
    total_red   = sum(d for s, d in zip(signals, durs) if s == 'RED')
    total_green = sum(d for s, d in zip(signals, durs) if s == 'GREEN')
    cycle_c = (total_green + total_red) / N_red if N_red else 0.0   # c
    green_g = total_green / N_red if N_red else 0.0                  # g
    red_r   = total_red   / N_red if N_red else 0.0                  # r  (avg RED per cycle)
    green_ratio_u = green_g / cycle_c if cycle_c else 0.0            # u

    # ════════════════════════════════════════════════════════════════════════════
    #  SECTION 3 — PCU
    #  Class counts taken from the Flow-Summary sheet (falls back to Speed-Log volumes)
    # ════════════════════════════════════════════════════════════════════════════
    summary_counts = {}
    for ws in wb.worksheets:
        if ws.title.lower().startswith('summary'):
            for r in ws.iter_rows(values_only=True):
                if r and r[0] and isinstance(r[0], str) and r[0].lower() in PCU_FACTOR \
                   and len(r) > 1 and isinstance(r[1], (int, float)):
                    summary_counts[r[0].lower()] = r[1]
            if summary_counts:
                break
    pcu_counts = summary_counts if summary_counts else dict(volume)
    total_pcu = sum(pcu_counts.get(k, 0) * PCU_FACTOR.get(k, 0.0) for k in PCU_FACTOR)

    # ════════════════════════════════════════════════════════════════════════════
    #  SECTION 4 — DELAY  (Hakkert & Gitelman railway level-crossing model)
    #
    #      T = Σ_{i=1..m, j=1..k}  ½ · t_el(i) · [ t_el(i) + t_rel(i,j) ] · λ(j) · n(i,j)
    #
    #  A single 15-minute video ⇒ one blockage category (m = 1) and one traffic
    #  interval (k = 1), so the double sum collapses to a single i=j=1 term with:
    #      t_el  = total RED  period   — gate-blockage time          (s)
    #      t_rel = total GREEN period  — queue-release time          (s)
    #      λ     = vehicle arrival rate from Speed Log-ARRIVAL (Section 1). It is
    #              applied as q = λ/3600 veh/s so that s · s · (veh/s) =
    #              vehicle-seconds (dimensionally consistent with the RED/GREEN
    #              periods, which are in seconds).
    #      n     = number of RED periods (gate closures) = N_red
    #
    #      T   = ½ · t_el · (t_el + t_rel) · q · n     [vehicle-seconds]
    #      T_d = T ÷ vehicle_count ÷ 60                [minutes / vehicle]
    #      d   = T_d · 60                              [seconds / vehicle]
    #  The formula uses the RED/GREEN periods directly.
    # ════════════════════════════════════════════════════════════════════════════
    t_el_red    = total_red       # blockage time  t_el   (total RED period, s)
    t_rel_green = total_green     # release time   t_rel  (total GREEN period, s)
    n_blockages = N_red           # n — number of RED periods (gate closures)
    T_delay  = 0.5 * t_el_red * (t_el_red + t_rel_green) * q_vps * n_blockages   # veh-seconds
    T_delay_vehh = T_delay / 3600.0                                              # veh-hours (info)
    T_d_min  = (T_delay / veh_total / 60.0) if veh_total else 0.0   # min/veh (count: SERVICE)
    d_s_veh  = T_d_min * 60.0                                                      # s/veh

    # ════════════════════════════════════════════════════════════════════════════
    #  SECTION 5 — TRAVEL-TIME COST  (HARD-CODED congestion threshold)
    #  (t_f − t_n) = avg traversal time of congested vehicles, where "congested"
    #  means Speed Log-SERVICE speed < CONG_SPEED_KMH (a hard-coded constant set
    #  in the config block above — no Otsu / percentile / median auto-detection).
    #
    #  PROVISION — NO EFFECTIVE RED PERIOD:
    #  If the gate analysis (Section 2) found no RED period (N_red == 0 or
    #  total RED time == 0 s), there is no rail-gate blockage in this window,
    #  hence no blockage-induced congestion to price. In that case
    #  (t_f − t_n) = 0 and ALL travel-time costs are 0 by construction.
    #
    #  Travel-time cost_i = (t_f − t_n)[min] · VOT_i · V_i
    # ════════════════════════════════════════════════════════════════════════════
    # ─── no-RED-period check (uses Section 2/4 gate results) ────────────────
    HAS_RED = (n_blockages > 0) and (t_el_red > 0)   # N_red and total RED seconds
    CONG_METHOD = f"hard-coded threshold < {CONG_SPEED_KMH:g} km/h"
    if HAS_RED:
        cong = [traversal_times[i] for i in range(len(traversal_times))
                if veh_speeds[i] is not None and veh_speeds[i] <= CONG_SPEED_KMH]
        tfn_sec = (sum(cong) / len(cong)) if cong else 0.0
    else:
        cong    = []
        tfn_sec = 0.0
    tfn_min = tfn_sec / 60.0
    travel_cost = {cl: tfn_min * VOT.get(cl, 0) * v for cl, v in volume.items()}
    if HAS_RED:
        print(f"Congestion threshold     : < {CONG_SPEED_KMH:g} km/h  [hard-coded]  "
              f"→ {len(cong)} congested vehicles")
    else:
        print("No effective RED period in this window → travel-time cost set to 0 "
              f"(N_red = {n_blockages}, total RED = {t_el_red:.1f} s)")

    # ════════════════════════════════════════════════════════════════════════════
    #  SECTION 6 — FUEL-LOSS COST
    #  Fuel cost_i = FC_i[ml/min] · T_d[min] · V_i · P_i[BDT/L] / 1000   (ml→L)
    # ════════════════════════════════════════════════════════════════════════════
    fuel_cost = {cl: FUEL_ML_MIN.get(cl, 0.0) * T_d_min * v * FUEL_PRICE.get(cl, 0.0) / 1000.0
                 for cl, v in volume.items()}

    # ════════════════════════════════════════════════════════════════════════════
    #  SECTION 7 — ELECTRICITY COST  (battery rickshaws only)
    #  C_elec = EC_idling[Wh/min] · T_d[min] · V_er · P_e[BDT/kWh] / 1000  (Wh→kWh)
    # ════════════════════════════════════════════════════════════════════════════
    elec_cost = {cl: (EC_IDLING_WH_MIN * T_d_min * v * P_ELEC_BDT_KWH / 1000.0)
                     if cl == 'rickshaw' else 0.0
                 for cl, v in volume.items()}

    # ════════════════════════════════════════════════════════════════════════════
    #  SECTION 8 — POLLUTION COST
    #  Cost_i = V_i · RL[km] · T_d[min] · Σ_j ( EF_{i,j}[g/km] · CP_j[BDT/kg] ) / 1000
    #  RL is the HARD-CODED road length (ROAD_LENGTH_M = 30 m = 0.03 km);
    #  /1000 converts g→kg.
    # ════════════════════════════════════════════════════════════════════════════
    RL_km = ROAD_LENGTH_M / 1000.0
    def _emission_cost_per_km(cl):
        cat = VEH_TO_EF.get(cl)
        if not cat:
            return 0.0
        return sum(EF[p].get(cat, 0.0) * CP[p] for p in POLLUTANTS)   # BDT per (g/km)·... aggregate
    pollution_cost = {cl: v * RL_km * T_d_min * _emission_cost_per_km(cl) / 1000.0
                      for cl, v in volume.items()}

    # ════════════════════════════════════════════════════════════════════════════
    #  SECTION 9 — VEHICLE OPERATING COST (VOC)  — for the 15-MINUTE video window
    #
    #      VOC_i = (VOC rate in Tk/km) × (average speed in km/h) × (delay in hr)
    #
    #  • VOC rate  : Projected Financial VOC at IRI 14, FY 2025-26 (Tk/km) — table
    #  • avg speed : per-class average of 'Speed (km/h)' from the SPEED LOG
    #  • delay (hr): class delay = T_d [min/veh] × V_i (total count of that class),
    #                exactly as done in the fuel / electricity / pollution costs,
    #                then ÷ 60 to convert minutes → hours.
    #  All quantities come from the same 15-minute observation window, so VOC_i is
    #  the operating cost incurred in those 15 minutes.
    # ════════════════════════════════════════════════════════════════════════════
    voc_delay_min = {cl: T_d_min * v for cl, v in volume.items()}            # min
    voc_delay_hr  = {cl: m / 60.0 for cl, m in voc_delay_min.items()}        # hr
    voc_cost = {cl: VOC_TK_KM.get(cl, 0.0) * avg_speed.get(cl, 0.0) * voc_delay_hr[cl]
                for cl in volume}

    # ════════════════════════════════════════════════════════════════════════════
    #  SECTION 10 — TOTAL-COST AGGREGATION  (all five costs merged)
    # ════════════════════════════════════════════════════════════════════════════
    classes_sorted = sorted(volume.keys(), key=lambda c: volume[c], reverse=True)
    cost_table = OrderedDict()
    for cl in classes_sorted:
        tt = travel_cost.get(cl, 0.0); fu = fuel_cost.get(cl, 0.0)
        el = elec_cost.get(cl, 0.0);   po = pollution_cost.get(cl, 0.0)
        vo = voc_cost.get(cl, 0.0)
        cost_table[cl] = (tt, fu, el, po, vo, tt + fu + el + po + vo)
    grand = [sum(cost_table[cl][k] for cl in cost_table) for k in range(6)]

    # ════════════════════════════════════════════════════════════════════════════
    #  SECTION 12 — QUEUE ANALYSIS  (formulas from the attached reference)
    #     (1)  T_Q = μ·r / (μ − λ)      Queue Duration Time (s)
    #     (2)  Q_M = λ·r / 3600         Maximum Queue Length (LVU)
    #     (3)  Q_L = Q_M × 20 / L_M     Vehicle Queue Length (m)
    #  λ = arrival rate (LVU/h, Speed Log-ARRIVAL); μ = service rate (LVU/h,
    #  Speed Log-SERVICE); r = effective RED period (s) from Section 2;
    #  L_M = ROAD_ENTRANCE_WIDTH_M (hard-coded). Computed for EACH RED event
    #  and for the average RED per cycle (r̄).
    # ════════════════════════════════════════════════════════════════════════════
    red_events = [d for s, d in runs if s == 'RED']      # individual RED durations (s)
    queue_oversaturated = not (mu_lvu_h > lam_lvu_h)     # formula (1) needs μ > λ

    def _queue_metrics(r_sec):
        QM = lam_lvu_h * r_sec / 3600.0                          # (2) LVU
        QL = QM * QL_M_PER_LVU / ROAD_ENTRANCE_WIDTH_M \
             if ROAD_ENTRANCE_WIDTH_M else 0.0                   # (3) m
        TQ = (mu_lvu_h * r_sec / (mu_lvu_h - lam_lvu_h)) \
             if not queue_oversaturated else None                # (1) s
        return TQ, QM, QL

    queue_rows = [(k + 1, r, *_queue_metrics(r)) for k, r in enumerate(red_events)]
    TQ_avg, QM_avg, QL_avg = _queue_metrics(red_r)        # r̄ = avg RED per cycle

    # ──────────────────────────────────────────────────────────────────────────
    #  CONSOLE SUMMARY
    # ──────────────────────────────────────────────────────────────────────────
    print("\n────────── TRAFFIC PARAMETERS ──────────")
    print(f"λ  arrival rate      : {lam_vph:10.2f} veh/h   (q = {q_vps:.4f} veh/s)  — Speed Log-ARRIVAL")
    print(f"   Arrival count     : {arr_total} veh over {obs_seconds:.1f} s")
    print(f"μ  service rate      : {mu_vph:10.2f} veh/h   (= {veh_total} veh ÷ {srv_obs_seconds:.1f} s × 3600)  — Speed Log-SERVICE")
    print(f"   λ / μ in LVU/h    : {lam_lvu_h:.2f} / {mu_lvu_h:.2f}")
    print(f"   gate cycle c/g/r  : {cycle_c:.1f} / {green_g:.1f} / {red_r:.1f} s   (u = {green_ratio_u:.3f})")
    print(f"   RED / GREEN total : {total_red:.1f} / {total_green:.1f} s   "
          f"(RED periods n = {n_blockages})")
    print(f"   T  total delay    : {T_delay:10.2f} veh-s  ({T_delay_vehh:.2f} veh-h)")
    print(f"   T_d / d           : {T_d_min:.4f} min/veh  ({d_s_veh:.2f} s/veh)")
    print(f"   Total PCU         : {total_pcu:10.2f}")

    print("\n────────── QUEUE ANALYSIS (Section 12) ──────────")
    if queue_oversaturated:
        print(f"⚠️  μ ≤ λ ({mu_lvu_h:.2f} ≤ {lam_lvu_h:.2f} LVU/h) — oversaturated: "
              "T_Q undefined (queue does not dissipate); Q_M and Q_L still reported.")
    for k, r_, TQ_, QM_, QL_ in queue_rows:
        print(f"   RED event {k}: r = {r_:7.1f} s   "
              f"T_Q = {('%.1f s' % TQ_) if TQ_ is not None else 'N/A (μ≤λ)':>12s}   "
              f"Q_M = {QM_:7.2f} LVU   Q_L = {QL_:7.2f} m")
    print(f"   Average (r̄ = {red_r:.1f} s): "
          f"T_Q = {('%.1f s' % TQ_avg) if TQ_avg is not None else 'N/A (μ≤λ)'}   "
          f"Q_M = {QM_avg:.2f} LVU   Q_L = {QL_avg:.2f} m")

    print("\n────────── COST SUMMARY (BDT, 15-min window) ──────────")
    print(f"{'Class':10s}{'TravelTime':>13s}{'Fuel':>13s}{'Electric':>13s}{'Pollution':>13s}{'VOC':>13s}{'Total':>13s}")
    for cl in cost_table:
        tt, fu, el, po, vo, to = cost_table[cl]
        print(f"{cl:10s}{tt:13.2f}{fu:13.2f}{el:13.2f}{po:13.2f}{vo:13.2f}{to:13.2f}")
    print(f"{'GRAND':10s}{grand[0]:13.2f}{grand[1]:13.2f}{grand[2]:13.2f}{grand[3]:13.2f}{grand[4]:13.2f}{grand[5]:13.2f}")

    # ════════════════════════════════════════════════════════════════════════════
    #  WRITE ANALYSIS SHEETS INTO THE WORKBOOK (originals preserved)
    # ════════════════════════════════════════════════════════════════════════════
    HDR_BG, SUB_BG, TOT_BG, WARN_BG = "1F3864", "2E75B6", "2E75B6", "C00000"
    thin = Border(*[Side(style='thin')] * 4)

    def H(c, bg=HDR_BG):
        c.font = Font(bold=True, color="FFFFFF", name='Arial', size=11)
        c.fill = PatternFill("solid", start_color=bg)
        c.alignment = Alignment(horizontal='center', vertical='center'); c.border = thin

    def D(c, alt=False, bold=False, num=None, align='center'):
        c.font = Font(name='Arial', size=10, bold=bold)
        c.fill = PatternFill("solid", start_color="DCE6F1" if alt else "FFFFFF")
        c.alignment = Alignment(horizontal=align, vertical='center'); c.border = thin
        if num:
            c.number_format = num

    def title(ws, text, span):
        ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=span)
        ws["A1"] = text
        ws["A1"].font = Font(bold=True, size=14, color="1F3864", name='Arial')
        ws["A1"].alignment = Alignment(horizontal='center')

    def drop_existing(name):
        if name in wb.sheetnames:
            del wb[name]

    def formula_note(ws, row, span, text, h=None):
        ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=span)
        c = ws.cell(row, 1); c.value = text
        c.font = Font(italic=True, name='Arial', size=10, color="555555")
        c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
        if h:
            ws.row_dimensions[row].height = h

    def total_row_style(ws, row, ncols, money_cols=()):
        for col in range(1, ncols + 1):
            c = ws.cell(row, col)
            c.font = Font(bold=True, color="FFFFFF", name='Arial', size=11)
            c.fill = PatternFill("solid", start_color=TOT_BG)
            c.alignment = Alignment(horizontal=('left' if col == 1 else 'center'),
                                    vertical='center')
            c.border = thin
            if col in money_cols:
                c.number_format = MONEY

    MONEY = '#,##0.00'
    NUM4  = '#,##0.0000'

    # ── Sheet A: Traffic Parameters (Sections 1–3) ──
    drop_existing("Traffic Parameters")
    wsA = wb.create_sheet("Traffic Parameters")
    title(wsA, "Traffic Parameters  (Sections 1–3)", 4)
    for col, h in enumerate(["Section", "Parameter", "Value", "Unit"], 1):
        H(wsA.cell(3, col)); wsA.cell(3, col).value = h
    paramA = [
     ("1", "Arrival vehicle count (Speed Log-Arrival)", arr_total, "veh"),
     ("1", "Observed time span (Speed Log-Arrival)", round(obs_seconds, 2), "s"),
     ("1", "Arrival rate λ  (= count ÷ span × 3600)", round(lam_vph, 4), "veh/h"),
     ("1", "Arrival rate q = λ/3600", round(q_vps, 6), "veh/s"),
     ("1", "Arrival rate λ (LVU-weighted)", round(lam_lvu_h, 4), "LVU/h"),
     ("1", "Service vehicle count (Speed Log-Service)", veh_total, "veh"),
     ("1", "Total counting time (Speed Log-Service)", round(srv_obs_seconds, 2), "s"),
     ("1", "Service rate μ  (= count ÷ counting time × 3600)", round(mu_vph, 4), "veh/h"),
     ("1", "Service rate μ (LVU-weighted)", round(mu_lvu_h, 4), "LVU/h"),
     ("2", "RED-detection bin size W (hard-coded)", round(float(WIN_SEC), 2), "s"),
     ("2", "Distinct RED events  N_red (= n)", N_red, "events"),
     ("2", "Distinct GREEN events  N_green", N_green, "events"),
     ("2", "Total GREEN time", round(total_green, 2), "s"),
     ("2", "Total RED time", round(total_red, 2), "s"),
     ("2", "Average cycle length  c", round(cycle_c, 4), "s"),
     ("2", "Average green per cycle  g", round(green_g, 4), "s"),
     ("2", "Average red per cycle  r", round(red_r, 4), "s"),
     ("2", "Effective green ratio  u = g/c", round(green_ratio_u, 4), "ratio"),
     ("3", "Total PCU", round(total_pcu, 4), "PCU"),
    ]
    for i, (sec, name, val, unit) in enumerate(paramA):
        rr = 4 + i; alt = i % 2 == 1
        D(wsA.cell(rr, 1), alt); wsA.cell(rr, 1).value = sec
        D(wsA.cell(rr, 2), alt, align='left'); wsA.cell(rr, 2).value = name
        D(wsA.cell(rr, 3), alt); wsA.cell(rr, 3).value = val
        D(wsA.cell(rr, 4), alt); wsA.cell(rr, 4).value = unit
    formula_note(wsA, 4 + len(paramA) + 1, 4,
                 "Section 1: λ = (Speed Log-ARRIVAL count ÷ observed span) × 3600; "
                 "μ = (Speed Log-SERVICE count ÷ total counting time) × 3600. "
                 f"Section 2: RED/GREEN is detected from the SERVICE exit times "
                 f"(fixed hard-coded bin size W = {WIN_SEC:g} s). LVU rates use the "
                 "LVU_FACTOR table (see the Queue Duration Time sheet).",
                 h=45)
    for col, w in zip('ABCD', [10, 44, 18, 18]):
        wsA.column_dimensions[col].width = w

    # ── Sheet B: Delay Estimation (Section 4) ──
    drop_existing("Delay Estimation")
    wsB = wb.create_sheet("Delay Estimation")
    title(wsB, "Delay Estimation — Hakkert & Gitelman (Section 4)", 3)
    wsB.merge_cells("A3:C3")
    wsB["A3"] = "T = ½ · t_el · (t_el + t_rel) · λ · n      (m = 1 train category, k = 1 interval)"
    wsB["A3"].font = Font(italic=True, name='Arial', size=10, color="555555")
    wsB["A3"].alignment = Alignment(horizontal='center')
    for col, h in enumerate(["Quantity", "Value", "Unit"], 1):
        H(wsB.cell(5, col)); wsB.cell(5, col).value = h
    paramB = [
     ("t_el — total RED (blockage) time", round(t_el_red, 4), "s"),
     ("t_rel — total GREEN (release) time", round(t_rel_green, 4), "s"),
     ("t_el + t_rel", round(t_el_red + t_rel_green, 4), "s"),
     ("λ — vehicle arrival rate (Speed Log-Arrival)", round(lam_vph, 4), "veh/h"),
     ("q = λ / 3600  (value used in formula)", round(q_vps, 6), "veh/s"),
     ("n — number of RED periods", n_blockages, "events"),
     ("T — total delay = ½·t_el·(t_el+t_rel)·q·n", round(T_delay, 4), "veh-s"),
     ("T — total delay (in vehicle-hours)", round(T_delay_vehh, 6), "veh-h"),
     ("Total vehicle count (Speed Log-Service)", veh_total, "veh"),
     ("T_d — average delay per vehicle", round(T_d_min, 6), "min/veh"),
     ("d  — average delay per vehicle", round(d_s_veh, 4), "s/veh"),
    ]
    for i, (name, val, unit) in enumerate(paramB):
        rr = 6 + i; alt = i % 2 == 1; bold = name.startswith(("T ", "T_d", "d  "))
        D(wsB.cell(rr, 1), alt, bold=bold, align='left'); wsB.cell(rr, 1).value = name
        D(wsB.cell(rr, 2), alt, bold=bold); wsB.cell(rr, 2).value = val
        D(wsB.cell(rr, 3), alt); wsB.cell(rr, 3).value = unit
    nB = 6 + len(paramB) + 1
    formula_note(wsB, nB, 3,
                 f"λ = {lam_vph:.2f} veh/h (counted from Speed Log-ARRIVAL) is applied as "
                 f"q = λ/3600 = {q_vps:.4f} veh/s so the product (s · s · veh/s) is in "
                 "vehicle-seconds. t_el and t_rel are the total RED and GREEN periods from "
                 "the auto-binned arrival timeline (Section 2); n is the number of RED "
                 "periods. The vehicle count dividing T comes from Speed Log-SERVICE.", h=58)
    for col, w in zip('ABC', [42, 18, 12]):
        wsB.column_dimensions[col].width = w

    # ── Sheet C: Travel Time Cost — detailed (Section 5) ──
    drop_existing("Travel Time Cost")
    wsC1 = wb.create_sheet("Travel Time Cost")
    title(wsC1, "Travel-Time Cost — detailed calculation (Section 5, BDT)", 6)
    formula_note(wsC1, 2, 6,
                 "Travel-time cost_i = (t_f − t_n) [min] × VOT_i [BDT/min] × V_i.   "
                 "(t_f − t_n) = average traversal time of congested vehicles in "
                 "Speed Log-SERVICE; the congestion speed threshold is HARD-CODED "
                 "(CONG_SPEED_KMH, set in the config block). PROVISION: if Section 2 "
                 "found NO effective RED period, "
                 "(t_f − t_n) = 0 and every travel-time cost is 0.", h=45)
    inp = [("Effective RED period present?", "YES" if HAS_RED else "NO — costs forced to 0", "—"),
           ("Congestion threshold (hard-coded)", round(CONG_SPEED_KMH, 4), "km/h"),
           ("Threshold method", CONG_METHOD, "—"),
           ("Congested vehicles (speed ≤ threshold)", len(cong), "veh"),
           ("(t_f − t_n) — avg congested traversal", round(tfn_sec, 4), "s"),
           ("(t_f − t_n) in minutes", round(tfn_min, 6), "min")]
    for col, h in enumerate(["Input", "Value", "Unit"], 1):
        H(wsC1.cell(4, col)); wsC1.cell(4, col).value = h
    for i, (n_, v_, u_) in enumerate(inp):
        rr = 5 + i; alt = i % 2 == 1
        D(wsC1.cell(rr, 1), alt, align='left'); wsC1.cell(rr, 1).value = n_
        D(wsC1.cell(rr, 2), alt); wsC1.cell(rr, 2).value = v_
        D(wsC1.cell(rr, 3), alt); wsC1.cell(rr, 3).value = u_
    hC1 = 5 + len(inp) + 1
    for col, h in enumerate(["Vehicle Class", "Volume V_i", "(t_f − t_n) (min)",
                             "VOT (BDT/min)", "Cost = (t_f−t_n)×VOT×V_i (BDT)"], 1):
        H(wsC1.cell(hC1, col)); wsC1.cell(hC1, col).value = h
    for i, cl in enumerate(classes_sorted):
        rr = hC1 + 1 + i; alt = i % 2 == 1
        D(wsC1.cell(rr, 1), alt, align='left'); wsC1.cell(rr, 1).value = cl
        D(wsC1.cell(rr, 2), alt); wsC1.cell(rr, 2).value = volume[cl]
        D(wsC1.cell(rr, 3), alt, num=NUM4); wsC1.cell(rr, 3).value = round(tfn_min, 6)
        D(wsC1.cell(rr, 4), alt); wsC1.cell(rr, 4).value = VOT.get(cl, 0)
        D(wsC1.cell(rr, 5), alt, bold=True, num=MONEY); wsC1.cell(rr, 5).value = round(travel_cost.get(cl, 0.0), 4)
    trC1 = hC1 + 1 + len(classes_sorted)
    wsC1.cell(trC1, 1).value = "TOTAL"; wsC1.cell(trC1, 2).value = sum(volume.values())
    wsC1.cell(trC1, 5).value = round(grand[0], 4)
    total_row_style(wsC1, trC1, 5, money_cols=(5,))
    for col, w in zip('ABCDE', [16, 12, 18, 16, 28]):
        wsC1.column_dimensions[col].width = w

    # ── Sheet D: Fuel Cost — detailed (Section 6) ──
    drop_existing("Fuel Cost")
    wsC2 = wb.create_sheet("Fuel Cost")
    title(wsC2, "Fuel-Loss Cost — detailed calculation (Section 6, BDT)", 9)
    formula_note(wsC2, 2, 9,
                 "Fuel cost_i = FC_i [ml/min] × T_d [min/veh] × V_i × P_i [BDT/L] ÷ 1000  (ml → L). "
                 f"T_d = {T_d_min:.6f} min/veh from the Delay Estimation sheet; "
                 "class delay = T_d × V_i (total count of that class).", h=30)
    hC2 = 4
    for col, h in enumerate(["Vehicle Class", "Volume V_i", "FC (ml/min)", "T_d (min/veh)",
                             "Class delay T_d×V_i (min)", "Fuel used (L)", "Fuel Type",
                             "Price (BDT/L)", "Cost (BDT)"], 1):
        H(wsC2.cell(hC2, col)); wsC2.cell(hC2, col).value = h
    for i, cl in enumerate(classes_sorted):
        rr = hC2 + 1 + i; alt = i % 2 == 1
        fc = FUEL_ML_MIN.get(cl, 0.0)
        d_min = T_d_min * volume[cl]
        litres = fc * d_min / 1000.0
        D(wsC2.cell(rr, 1), alt, align='left'); wsC2.cell(rr, 1).value = cl
        D(wsC2.cell(rr, 2), alt); wsC2.cell(rr, 2).value = volume[cl]
        D(wsC2.cell(rr, 3), alt); wsC2.cell(rr, 3).value = fc
        D(wsC2.cell(rr, 4), alt, num=NUM4); wsC2.cell(rr, 4).value = round(T_d_min, 6)
        D(wsC2.cell(rr, 5), alt, num=NUM4); wsC2.cell(rr, 5).value = round(d_min, 4)
        D(wsC2.cell(rr, 6), alt, num=NUM4); wsC2.cell(rr, 6).value = round(litres, 6)
        D(wsC2.cell(rr, 7), alt, align='left'); wsC2.cell(rr, 7).value = FUEL_LABEL.get(cl, '—')
        D(wsC2.cell(rr, 8), alt); wsC2.cell(rr, 8).value = FUEL_PRICE.get(cl, 0.0)
        D(wsC2.cell(rr, 9), alt, bold=True, num=MONEY); wsC2.cell(rr, 9).value = round(fuel_cost.get(cl, 0.0), 4)
    trC2 = hC2 + 1 + len(classes_sorted)
    wsC2.cell(trC2, 1).value = "TOTAL"; wsC2.cell(trC2, 2).value = sum(volume.values())
    wsC2.cell(trC2, 9).value = round(grand[1], 4)
    total_row_style(wsC2, trC2, 9, money_cols=(9,))
    for col, w in zip('ABCDEFGHI', [16, 12, 12, 14, 22, 14, 18, 14, 16]):
        wsC2.column_dimensions[col].width = w

    # ── Sheet E: Electricity Cost — detailed (Section 7) ──
    drop_existing("Electricity Cost")
    wsC3 = wb.create_sheet("Electricity Cost")
    title(wsC3, "Electricity Cost — detailed calculation (Section 7, BDT)", 8)
    formula_note(wsC3, 2, 8,
                 "C_elec = EC_idling [Wh/min] × T_d [min/veh] × V_i × P_e [BDT/kWh] ÷ 1000 "
                 "(Wh → kWh). Applies to battery rickshaws only; all other classes "
                 "consume no electricity.", h=30)
    hC3 = 4
    for col, h in enumerate(["Vehicle Class", "Volume V_i", "EC idling (Wh/min)",
                             "T_d (min/veh)", "Class delay T_d×V_i (min)",
                             "Energy (kWh)", "Price (BDT/kWh)", "Cost (BDT)"], 1):
        H(wsC3.cell(hC3, col)); wsC3.cell(hC3, col).value = h
    for i, cl in enumerate(classes_sorted):
        rr = hC3 + 1 + i; alt = i % 2 == 1
        ec = EC_IDLING_WH_MIN if cl == 'rickshaw' else 0.0
        d_min = T_d_min * volume[cl]
        kwh = ec * d_min / 1000.0
        D(wsC3.cell(rr, 1), alt, align='left'); wsC3.cell(rr, 1).value = cl
        D(wsC3.cell(rr, 2), alt); wsC3.cell(rr, 2).value = volume[cl]
        D(wsC3.cell(rr, 3), alt); wsC3.cell(rr, 3).value = ec
        D(wsC3.cell(rr, 4), alt, num=NUM4); wsC3.cell(rr, 4).value = round(T_d_min, 6)
        D(wsC3.cell(rr, 5), alt, num=NUM4); wsC3.cell(rr, 5).value = round(d_min, 4)
        D(wsC3.cell(rr, 6), alt, num=NUM4); wsC3.cell(rr, 6).value = round(kwh, 6)
        D(wsC3.cell(rr, 7), alt); wsC3.cell(rr, 7).value = P_ELEC_BDT_KWH if cl == 'rickshaw' else 0.0
        D(wsC3.cell(rr, 8), alt, bold=True, num=MONEY); wsC3.cell(rr, 8).value = round(elec_cost.get(cl, 0.0), 4)
    trC3 = hC3 + 1 + len(classes_sorted)
    wsC3.cell(trC3, 1).value = "TOTAL"; wsC3.cell(trC3, 2).value = sum(volume.values())
    wsC3.cell(trC3, 8).value = round(grand[2], 4)
    total_row_style(wsC3, trC3, 8, money_cols=(8,))
    for col, w in zip('ABCDEFGH', [16, 12, 18, 14, 22, 14, 16, 16]):
        wsC3.column_dimensions[col].width = w

    # ── Sheet F: Pollution Cost — detailed (Section 8) ──
    drop_existing("Pollution Cost")
    wsC4 = wb.create_sheet("Pollution Cost")
    title(wsC4, "Pollution Cost — detailed calculation (Section 8, BDT)", 8)
    formula_note(wsC4, 2, 8,
                 "Cost_i = V_i × RL [km] × T_d [min/veh] × Σ_j (EF_ij [g/km] × CP_j [BDT/kg]) ÷ 1000 "
                 f"(g → kg).  RL is HARD-CODED: ROAD_LENGTH_M = {ROAD_LENGTH_M:g} m = {RL_km:g} km. "
                 "Rickshaw (battery) and bicycle have no tailpipe emissions.", h=30)
    hC4 = 4
    for col, h in enumerate(["Vehicle Class", "EF Category", "Volume V_i", "RL (km)",
                             "T_d (min/veh)", "Class delay T_d×V_i (min)",
                             "Σ EF×CP (BDT/kg · g/km)", "Cost (BDT)"], 1):
        H(wsC4.cell(hC4, col)); wsC4.cell(hC4, col).value = h
    for i, cl in enumerate(classes_sorted):
        rr = hC4 + 1 + i; alt = i % 2 == 1
        cat = VEH_TO_EF.get(cl, '—' if cl not in ('rickshaw', 'bicycle') else 'none')
        efcp = _emission_cost_per_km(cl)
        D(wsC4.cell(rr, 1), alt, align='left'); wsC4.cell(rr, 1).value = cl
        D(wsC4.cell(rr, 2), alt); wsC4.cell(rr, 2).value = VEH_TO_EF.get(cl, 'none')
        D(wsC4.cell(rr, 3), alt); wsC4.cell(rr, 3).value = volume[cl]
        D(wsC4.cell(rr, 4), alt); wsC4.cell(rr, 4).value = RL_km
        D(wsC4.cell(rr, 5), alt, num=NUM4); wsC4.cell(rr, 5).value = round(T_d_min, 6)
        D(wsC4.cell(rr, 6), alt, num=NUM4); wsC4.cell(rr, 6).value = round(T_d_min * volume[cl], 4)
        D(wsC4.cell(rr, 7), alt, num=MONEY); wsC4.cell(rr, 7).value = round(efcp, 4)
        D(wsC4.cell(rr, 8), alt, bold=True, num=MONEY); wsC4.cell(rr, 8).value = round(pollution_cost.get(cl, 0.0), 4)
    trC4 = hC4 + 1 + len(classes_sorted)
    wsC4.cell(trC4, 1).value = "TOTAL"; wsC4.cell(trC4, 3).value = sum(volume.values())
    wsC4.cell(trC4, 8).value = round(grand[3], 4)
    total_row_style(wsC4, trC4, 8, money_cols=(8,))
    # per-pollutant breakdown
    brk = trC4 + 2
    wsC4.merge_cells(start_row=brk, start_column=1, end_row=brk, end_column=8)
    cbk = wsC4.cell(brk, 1); cbk.value = "Per-pollutant cost breakdown (BDT) — cost_ij = V_i × RL × T_d × EF_ij × CP_j ÷ 1000"
    cbk.font = Font(bold=True, color="FFFFFF", name='Arial', size=10)
    cbk.fill = PatternFill("solid", start_color=SUB_BG)
    cbk.alignment = Alignment(horizontal='left', vertical='center')
    brk += 1
    for col, h in enumerate(["Vehicle Class"] + POLLUTANTS, 1):
        H(wsC4.cell(brk, col)); wsC4.cell(brk, col).value = h
    for i, cl in enumerate(classes_sorted):
        rr = brk + 1 + i; alt = i % 2 == 1
        D(wsC4.cell(rr, 1), alt, align='left'); wsC4.cell(rr, 1).value = cl
        cat = VEH_TO_EF.get(cl)
        for j, p in enumerate(POLLUTANTS):
            v = (volume[cl] * RL_km * T_d_min * EF[p].get(cat, 0.0) * CP[p] / 1000.0) if cat else 0.0
            D(wsC4.cell(rr, 2 + j), alt, num=MONEY); wsC4.cell(rr, 2 + j).value = round(v, 4)
    for col, w in zip('ABCDEFGHI', [16, 12, 12, 10, 14, 22, 22, 16, 12]):
        wsC4.column_dimensions[col].width = w

    # ── Sheet G: VOC — detailed (Section 9) ──
    drop_existing("VOC")
    wsC5 = wb.create_sheet("VOC")
    title(wsC5, f"Vehicle Operating Cost (VOC) — {VOC_WINDOW_MIN:g}-minute window (Section 9, BDT)", 8)
    formula_note(wsC5, 2, 8,
                 "VOC_i = (VOC rate, Tk/km) × (average speed, km/h) × (delay, hr).  "
                 "VOC rate: Projected Financial VOC at IRI 14, FY 2025-26 (Tk/km). "
                 "Average speed: per-class mean of 'Speed (km/h)' from SPEED LOG-SERVICE. "
                 "Delay (hr) = T_d [min/veh] × V_i (total count of that class) ÷ 60 — the same "
                 "T_d × V_i class delay used in the fuel / electricity / pollution costs. "
                 f"All inputs come from the same {VOC_WINDOW_MIN:g}-minute video window, so the "
                 f"VOC below is the operating cost for those {VOC_WINDOW_MIN:g} minutes.", h=58)
    hC5 = 4
    for col, h in enumerate(["Vehicle Class", "Volume V_i", "VOC rate (Tk/km)",
                             "Avg speed (km/h)", "T_d (min/veh)",
                             "Class delay T_d×V_i (min)", "Delay (hr)",
                             "VOC = rate×speed×delay (BDT)"], 1):
        H(wsC5.cell(hC5, col)); wsC5.cell(hC5, col).value = h
    for i, cl in enumerate(classes_sorted):
        rr = hC5 + 1 + i; alt = i % 2 == 1
        D(wsC5.cell(rr, 1), alt, align='left'); wsC5.cell(rr, 1).value = cl
        D(wsC5.cell(rr, 2), alt); wsC5.cell(rr, 2).value = volume[cl]
        D(wsC5.cell(rr, 3), alt, num=MONEY); wsC5.cell(rr, 3).value = VOC_TK_KM.get(cl, 0.0)
        D(wsC5.cell(rr, 4), alt, num=NUM4); wsC5.cell(rr, 4).value = round(avg_speed.get(cl, 0.0), 4)
        D(wsC5.cell(rr, 5), alt, num=NUM4); wsC5.cell(rr, 5).value = round(T_d_min, 6)
        D(wsC5.cell(rr, 6), alt, num=NUM4); wsC5.cell(rr, 6).value = round(voc_delay_min[cl], 4)
        D(wsC5.cell(rr, 7), alt, num=NUM4); wsC5.cell(rr, 7).value = round(voc_delay_hr[cl], 6)
        D(wsC5.cell(rr, 8), alt, bold=True, num=MONEY); wsC5.cell(rr, 8).value = round(voc_cost.get(cl, 0.0), 4)
    trC5 = hC5 + 1 + len(classes_sorted)
    wsC5.cell(trC5, 1).value = "TOTAL"; wsC5.cell(trC5, 2).value = sum(volume.values())
    wsC5.cell(trC5, 8).value = round(grand[4], 4)
    total_row_style(wsC5, trC5, 8, money_cols=(8,))
    # VOC reference table (from the photo)
    ref = trC5 + 2
    wsC5.merge_cells(start_row=ref, start_column=1, end_row=ref, end_column=8)
    crf = wsC5.cell(ref, 1)
    crf.value = "Projected Financial VOC at IRI 14, FY 2025-26 (Taka/km) — source table"
    crf.font = Font(bold=True, color="FFFFFF", name='Arial', size=10)
    crf.fill = PatternFill("solid", start_color=SUB_BG)
    crf.alignment = Alignment(horizontal='left', vertical='center')
    ref += 1
    for col, h in enumerate(["Vehicle (source table)", "Detected class", "VOC 2026 (Tk/km)"], 1):
        H(wsC5.cell(ref, col)); wsC5.cell(ref, col).value = h
    for i, (cl, rate) in enumerate(VOC_TK_KM.items()):
        rr = ref + 1 + i; alt = i % 2 == 1
        D(wsC5.cell(rr, 1), alt, align='left'); wsC5.cell(rr, 1).value = VOC_SOURCE_LABEL[cl]
        D(wsC5.cell(rr, 2), alt); wsC5.cell(rr, 2).value = cl
        D(wsC5.cell(rr, 3), alt, num=MONEY); wsC5.cell(rr, 3).value = rate
    for col, w in zip('ABCDEFGH', [22, 12, 16, 16, 14, 22, 12, 28]):
        wsC5.column_dimensions[col].width = w

    # ── Sheet H: Cost Summary — all five costs merged (Section 10) ──
    drop_existing("Cost Summary")
    wsC = wb.create_sheet("Cost Summary")
    title(wsC, "Cost Summary  (Sections 5–10, BDT)", 8)
    headers = ["Vehicle Class", "Volume", "Travel-Time Cost", "Fuel-Loss Cost",
               "Electricity Cost", "Pollution Cost", "VOC", "Total Cost"]
    for col, h in enumerate(headers, 1):
        H(wsC.cell(3, col)); wsC.cell(3, col).value = h
    start = 4
    for i, cl in enumerate(cost_table):
        rr = start + i; alt = i % 2 == 1
        tt, fu, el, po, vo, to = cost_table[cl]
        D(wsC.cell(rr, 1), alt, align='left'); wsC.cell(rr, 1).value = cl
        D(wsC.cell(rr, 2), alt); wsC.cell(rr, 2).value = volume[cl]
        for k, val in enumerate((tt, fu, el, po, vo), start=3):
            D(wsC.cell(rr, k), alt, num=MONEY); wsC.cell(rr, k).value = round(val, 4)
        D(wsC.cell(rr, 8), alt, bold=True, num=MONEY); wsC.cell(rr, 8).value = round(to, 4)
    gr = start + len(cost_table)
    wsC.cell(gr, 1).value = "GRAND TOTAL"
    wsC.cell(gr, 2).value = sum(volume.values())
    for k, val in enumerate(grand[:5], start=3):
        wsC.cell(gr, k).value = round(val, 4)
    wsC.cell(gr, 8).value = round(grand[5], 4)
    total_row_style(wsC, gr, 8, money_cols=(3, 4, 5, 6, 7, 8))
    for col, w in zip('ABCDEFGH', [16, 10, 18, 16, 16, 16, 14, 16]):
        wsC.column_dimensions[col].width = w

    # ── Sheet I: Reference Factors (all lookup tables + mappings + unit notes) ──
    drop_existing("Reference Factors")
    wsD = wb.create_sheet("Reference Factors")
    title(wsD, "Reference Factors & Mapping", 6)
    row = 3
    def section_header(ws, r, text, span=6):
        ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=span)
        c = ws.cell(r, 1); c.value = text
        c.font = Font(bold=True, color="FFFFFF", name='Arial', size=11)
        c.fill = PatternFill("solid", start_color=SUB_BG)
        c.alignment = Alignment(horizontal='left', vertical='center')
        return r + 1

    # PCU
    row = section_header(wsD, row, "Section 3 — PCU factors")
    for col, h in enumerate(["Vehicle Class", "PCU Factor"], 1):
        H(wsD.cell(row, col)); wsD.cell(row, col).value = h
    row += 1
    for i, (k, v) in enumerate(PCU_FACTOR.items()):
        D(wsD.cell(row, 1), i % 2 == 1, align='left'); wsD.cell(row, 1).value = k
        D(wsD.cell(row, 2), i % 2 == 1); wsD.cell(row, 2).value = v
        row += 1
    row += 1
    # VOT
    row = section_header(wsD, row, "Section 5 — Value of Time (VOT, 2026)")
    for col, h in enumerate(["Vehicle Class", "VOT (BDT/min)"], 1):
        H(wsD.cell(row, col)); wsD.cell(row, col).value = h
    row += 1
    for i, (k, v) in enumerate(VOT.items()):
        D(wsD.cell(row, 1), i % 2 == 1, align='left'); wsD.cell(row, 1).value = k
        D(wsD.cell(row, 2), i % 2 == 1); wsD.cell(row, 2).value = v
        row += 1
    row += 1
    # Fuel
    row = section_header(wsD, row, "Section 6 — Idling fuel use, fuel type & price")
    for col, h in enumerate(["Vehicle Class", "Fuel (ml/min)", "Fuel Type", "Price (BDT/L)"], 1):
        H(wsD.cell(row, col)); wsD.cell(row, col).value = h
    row += 1
    for i, k in enumerate(FUEL_ML_MIN):
        D(wsD.cell(row, 1), i % 2 == 1, align='left'); wsD.cell(row, 1).value = k
        D(wsD.cell(row, 2), i % 2 == 1); wsD.cell(row, 2).value = FUEL_ML_MIN[k]
        D(wsD.cell(row, 3), i % 2 == 1, align='left'); wsD.cell(row, 3).value = FUEL_LABEL[k]
        D(wsD.cell(row, 4), i % 2 == 1); wsD.cell(row, 4).value = FUEL_PRICE[k]
        row += 1
    row += 1
    # Emission factors
    row = section_header(wsD, row, "Section 8 — Emission factors EF (g/km)")
    hdr = ["Pollutant"] + [c.upper() for c in ['bus','micro','tempo','car','cng','bike','lorry','truck','pickup']] + ["Cost (BDT/kg)"]
    for col, h in enumerate(hdr, 1):
        H(wsD.cell(row, col)); wsD.cell(row, col).value = h
    row += 1
    ef_cats = ['bus','micro','tempo','car','cng','bike','lorry','truck','pickup']
    for i, p in enumerate(POLLUTANTS):
        D(wsD.cell(row, 1), i % 2 == 1, align='left'); wsD.cell(row, 1).value = p
        for j, cat in enumerate(ef_cats):
            D(wsD.cell(row, 2 + j), i % 2 == 1); wsD.cell(row, 2 + j).value = EF[p].get(cat, 0)
        D(wsD.cell(row, 2 + len(ef_cats)), i % 2 == 1); wsD.cell(row, 2 + len(ef_cats)).value = CP[p]
        row += 1
    row += 1
    # VOC rates
    row = section_header(wsD, row, "Section 9 — Vehicle Operating Cost rates (Projected Financial VOC at IRI 14, FY 2025-26)")
    for col, h in enumerate(["Vehicle (source table)", "Detected class", "VOC 2026 (Tk/km)"], 1):
        H(wsD.cell(row, col)); wsD.cell(row, col).value = h
    row += 1
    for i, (k, v) in enumerate(VOC_TK_KM.items()):
        D(wsD.cell(row, 1), i % 2 == 1, align='left'); wsD.cell(row, 1).value = VOC_SOURCE_LABEL[k]
        D(wsD.cell(row, 2), i % 2 == 1); wsD.cell(row, 2).value = k
        D(wsD.cell(row, 3), i % 2 == 1); wsD.cell(row, 3).value = v
        row += 1
    row += 1
    # Class → EF mapping
    row = section_header(wsD, row, "Vehicle class → EF category mapping")
    for col, h in enumerate(["Vehicle Class", "EF Category"], 1):
        H(wsD.cell(row, col)); wsD.cell(row, col).value = h
    row += 1
    ef_map_full = dict(VEH_TO_EF); ef_map_full.update({'rickshaw': 'electric (none)', 'bicycle': 'none', 'van': 'none'})
    for i, (k, v) in enumerate(ef_map_full.items()):
        D(wsD.cell(row, 1), i % 2 == 1, align='left'); wsD.cell(row, 1).value = k
        D(wsD.cell(row, 2), i % 2 == 1, align='left'); wsD.cell(row, 2).value = v
        row += 1
    row += 1
    # Notes / assumptions
    row = section_header(wsD, row, "Assumptions & unit conversions")
    notes = [
     f"FPS = {FPS:g}.  Traversal time = (Exit Frame − Enter Frame) / FPS.",
     "Section 1: λ counted from SPEED LOG-ARRIVAL (count ÷ span × 3600); "
     "μ counted from SPEED LOG-SERVICE (count ÷ total counting time × 3600).",
     "Section 4 delay (Hakkert & Gitelman): T = ½·t_el·(t_el+t_rel)·λ·n, with "
     "t_el = total RED period, t_rel = total GREEN period, λ = arrival rate "
     "(applied as q = λ/3600 veh/s), n = number of RED periods.",
     "Fuel-loss cost divides by 1000 to convert ml → litre (price is BDT/litre).",
     "Electricity cost divides by 1000 to convert Wh → kWh (price is BDT/kWh).",
     f"Pollution: RL is HARD-CODED — ROAD_LENGTH_M = {ROAD_LENGTH_M:g} m = {RL_km:g} km; divide by 1000 to convert g → kg.",
     f"VOC = rate[Tk/km] × avg speed[km/h] × delay[hr]; calculated for the {VOC_WINDOW_MIN:g}-minute video window. "
     "Average speed per class from Speed Log-SERVICE; delay = T_d × V_i ÷ 60.",
     "Rickshaws are battery-powered: no fuel & no tailpipe emissions (electricity cost only).",
     "Bicycle and van consume no fuel (Section 6).",
     "All vehicle volumes, speeds and costs come from SPEED LOG-SERVICE; "
     "λ comes from SPEED LOG-ARRIVAL; the gate RED/GREEN timing comes from "
     "SPEED LOG-SERVICE (fixed hard-coded bin size).",
     f"Queue formulas (Section 12): T_Q = μ·r/(μ−λ); Q_M = λ·r/3600; Q_L = Q_M×{QL_M_PER_LVU:g}/L_M, "
     f"with L_M = {ROAD_ENTRANCE_WIDTH_M:g} m (hard-coded) and λ, μ in LVU/h.",
    ]
    for i, n in enumerate(notes):
        wsD.merge_cells(start_row=row, start_column=1, end_row=row, end_column=6)
        c = wsD.cell(row, 1); c.value = "•  " + n
        c.font = Font(name='Arial', size=10)
        c.alignment = Alignment(horizontal='left', vertical='center')
        row += 1
    for col, w in zip('ABCDEFGHIJK', [24, 14, 16, 13, 11, 11, 11, 11, 11, 11, 14]):
        wsD.column_dimensions[col].width = w

    # ════════════════════════════════════════════════════════════════════════════
    #  SECTION 11 — WHOLE-INTERSECTION ESTIMATE  (Directional Distribution Factor D)
    # ────────────────────────────────────────────────────────────────────────────
    #  This camera observes ONE direction across the shared rail gate. To estimate
    #  the whole intersection from a single-direction count, traffic-engineering
    #  practice (AASHTO Green Book; HCM 6th Ed., Ch. 19) uses the Directional
    #  Distribution Factor:
    #        D = (peak-direction volume) / (total two-way volume)
    #        Intersection value = per-camera value ÷ D
    #
    #  • COUNT-based quantities (λ, vehicle counts, ALL costs incl. VOC) scale by
    #    1/D, because they are proportional to the number of vehicles.
    #  • PER-VEHICLE quantities (delay d, T_d per veh) are UNCHANGED — they do not
    #    depend on how many directions are observed.
    #  • The shared rail-gate timing (Section 2) is already intersection-wide.
    #
    #  Results are reported as a SENSITIVITY RANGE over a plausible band of D — the
    #  standard, honest way to present a single-direction expansion.
    # ════════════════════════════════════════════════════════════════════════════

    # ─── tunable: directional distribution factor ───────────────────────────────
    #  D_MAIN  : best single estimate (AASHTO default 0.55 for a dominant direction)
    #  D_RANGE : (low, high) band for the sensitivity rows
    #            0.50 = balanced two-way flow  → scale 2.00×
    #            1.00 = one-way road (all flow seen) → scale 1.00× (no expansion)
    #  If you measure the opposing flow on another camera, set
    #  D_MAIN = this_dir_count / (this_dir_count + opposing_count) for an exact value.
    D_MAIN  = DDF_MAIN  if "DDF_MAIN"  in globals() else 0.55
    D_RANGE = DDF_RANGE if "DDF_RANGE" in globals() else (0.50, 0.60)
    _Dlo, _Dhi = min(D_RANGE), max(D_RANGE)
    _sf = (1.0 / D_MAIN) if D_MAIN else 1.0          # scale factor = 1/D

    # ─── intersection-level estimates at the central D ──────────────────────────
    lam_int   = lam_vph    / D_MAIN if D_MAIN else lam_vph     # arrival rate (÷D)
    veh_int   = veh_total  / D_MAIN if D_MAIN else veh_total   # vehicle count (÷D)
    # per-vehicle delay d_s_veh and T_d_min are UNCHANGED

    # cost components — every cost is count-based ⇒ all scale by 1/D
    travel_int = grand[0] / D_MAIN if D_MAIN else grand[0]
    fuel_int   = grand[1] / D_MAIN if D_MAIN else grand[1]
    elec_int   = grand[2] / D_MAIN if D_MAIN else grand[2]
    poll_int   = grand[3] / D_MAIN if D_MAIN else grand[3]
    voc_int    = grand[4] / D_MAIN if D_MAIN else grand[4]
    grand_int  = travel_int + fuel_int + elec_int + poll_int + voc_int

    # grand total at the two range endpoints (for the reported band)
    grand_int_lo = grand[5] / _Dhi if _Dhi else grand[5]   # high D → smallest expansion
    grand_int_hi = grand[5] / _Dlo if _Dlo else grand[5]   # low  D → largest  expansion

    # ─── console summary ─────────────────────────────────────────────────────────
    print("\n────────── WHOLE-INTERSECTION ESTIMATE (Section 11) ──────────")
    print(f"Directional factor D : {D_MAIN:.3f}   (range {_Dlo:g}–{_Dhi:g}, scale 1/D = {_sf:.3f})")
    print(f"λ  intersection      : {lam_int:10.2f} veh/h   (per-camera {lam_vph:.2f})")
    print(f"d per veh            : unchanged ({d_s_veh:.2f} s/veh)")
    print(f"Vehicles (Service)   : {veh_int:10.2f} veh     (per-camera {veh_total})")
    print(f"{'':10s}{'per-camera':>14s}{'intersection':>16s}")
    print(f"{'TravelTime':10s}{grand[0]:14.2f}{travel_int:16.2f}")
    print(f"{'Fuel':10s}{grand[1]:14.2f}{fuel_int:16.2f}")
    print(f"{'Electric':10s}{grand[2]:14.2f}{elec_int:16.2f}")
    print(f"{'Pollution':10s}{grand[3]:14.2f}{poll_int:16.2f}")
    print(f"{'VOC':10s}{grand[4]:14.2f}{voc_int:16.2f}")
    print(f"{'GRAND':10s}{grand[5]:14.2f}{grand_int:16.2f}   (at D = {D_MAIN:g})")
    print(f"Reported range       : {grand_int_lo:.2f} – {grand_int_hi:.2f} BDT "
          f"(D = {_Dhi:g} → {_Dlo:g})")

    # ─── write Section 11 sheet (reuses this cell's H / D / title / drop_existing) ─
    drop_existing("Intersection Estimate")
    wsI = wb.create_sheet("Intersection Estimate")
    title(wsI, "Whole-Intersection Estimate  (Directional Distribution)", 4)

    _r = 3
    def _hdr(cols):
        nonlocal _r
        for col, h in enumerate(cols, 1):
            H(wsI.cell(_r, col)); wsI.cell(_r, col).value = h
        _r += 1
    def _row(vals, alt, money_cols=()):
        nonlocal _r
        for col, v in enumerate(vals, 1):
            al = 'left' if col == 1 else 'center'
            D(wsI.cell(_r, col), alt, align=al, num=(MONEY if col in money_cols else None))
            wsI.cell(_r, col).value = v
        _r += 1
    def _band(text):
        nonlocal _r
        wsI.merge_cells(start_row=_r, start_column=1, end_row=_r, end_column=4)
        c = wsI.cell(_r, 1); c.value = text
        c.font = Font(bold=True, color="FFFFFF", name='Arial', size=10)
        c.fill = PatternFill("solid", start_color=SUB_BG)
        c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
        _r += 1

    _band("Method: Intersection value = per-camera value ÷ D   (AASHTO Green Book; HCM 6th Ed., Ch. 19). "
          "D = peak-direction volume ÷ total two-way volume. Count-based values scale by 1/D; "
          "per-vehicle values (delay d, T_d) are unchanged.")
    _r += 1

    _hdr(["Directional factor", "Value", "Unit", "Notes"])
    _row(["D (main estimate)", round(D_MAIN, 4), "—", "AASHTO default 0.55 (dominant direction)"], False)
    _row(["Scale factor 1/D", round(_sf, 4), "—", "Multiplier on count-based values"], True)
    _row(["D range (low–high)", f"{_Dlo:g} – {_Dhi:g}", "—", "Used for the sensitivity band"], False)
    _r += 1

    _hdr(["Parameter", "Per-camera", "Intersection (÷D)", "Notes"])
    _row(["Arrival rate λ (veh/h)", round(lam_vph, 2), round(lam_int, 2), "Count-based ⇒ ÷D"], False)
    _row(["Delay d (s/veh)", round(d_s_veh, 4), round(d_s_veh, 4), "Per-vehicle ⇒ unchanged"], True)
    _row(["T_d (min/veh)", round(T_d_min, 4), round(T_d_min, 4), "Per-vehicle ⇒ unchanged"], False)
    _row(["Vehicles (Speed Log-Service)", veh_total, round(veh_int, 2), "Count-based ⇒ ÷D"], True)
    _r += 1

    _hdr(["Cost component", "Per-camera (BDT)", "Intersection (BDT)", "Notes"])
    _row(["Travel time (Section 5)", round(grand[0], 4), round(travel_int, 4), "÷D"], False, money_cols=(2, 3))
    _row(["Fuel loss (Section 6)", round(grand[1], 4), round(fuel_int, 4), "÷D"], True, money_cols=(2, 3))
    _row(["Electricity (Section 7)", round(grand[2], 4), round(elec_int, 4), "÷D"], False, money_cols=(2, 3))
    _row(["Pollution (Section 8)", round(grand[3], 4), round(poll_int, 4), "÷D"], True, money_cols=(2, 3))
    _row(["VOC (Section 9)", round(grand[4], 4), round(voc_int, 4), "÷D"], False, money_cols=(2, 3))
    # grand-total row (highlighted)
    for col, v in enumerate(["INTERSECTION GRAND TOTAL", round(grand[5], 4), round(grand_int, 4),
                             f"at D = {D_MAIN:g}"], 1):
        cc = wsI.cell(_r, col); cc.value = v
        cc.font = Font(bold=True, color="FFFFFF", name='Arial', size=11)
        cc.fill = PatternFill("solid", start_color=TOT_BG)
        cc.alignment = Alignment(horizontal=('left' if col == 1 else 'center'), vertical='center')
        cc.border = thin
        if col in (2, 3):
            cc.number_format = MONEY
    _r += 2

    _hdr(["D value", "Scale 1/D", "Intersection grand total (BDT)", "Interpretation"])
    _row([round(_Dhi, 4), round(1.0/_Dhi, 4) if _Dhi else 0, round(grand_int_lo, 4),
          "High D → smallest expansion"], False, money_cols=(3,))
    _row([round(D_MAIN, 4), round(_sf, 4), round(grand_int, 4), "Main estimate"], True, money_cols=(3,))
    _row([round(_Dlo, 4), round(1.0/_Dlo, 4) if _Dlo else 0, round(grand_int_hi, 4),
          "Low D → largest expansion"], False, money_cols=(3,))
    for col, v in enumerate(["REPORTED RANGE", "",
                             f"{round(grand_int_lo, 2)} – {round(grand_int_hi, 2)} BDT",
                             "per observation window"], 1):
        cc = wsI.cell(_r, col); cc.value = v
        cc.font = Font(bold=True, color="FFFFFF", name='Arial', size=11)
        cc.fill = PatternFill("solid", start_color=TOT_BG)
        cc.alignment = Alignment(horizontal=('left' if col == 1 else 'center'), vertical='center')
        cc.border = thin
    _r += 1

    for col, w in zip("ABCD", [30, 16, 22, 42]):
        wsI.column_dimensions[col].width = w


    # ════════════════════════════════════════════════════════════════════════════
    #  SECTION 12 SHEETS — three separate sheets, one per queue formula
    # ════════════════════════════════════════════════════════════════════════════
    _QUEUE_COMMON_INPUTS = [
        ("λ — arrival rate (Speed Log-Arrival)", round(lam_lvu_h, 4), "LVU/h"),
        ("μ — service rate (Speed Log-Service)", round(mu_lvu_h, 4), "LVU/h"),
        ("μ − λ", round(mu_lvu_h - lam_lvu_h, 4), "LVU/h"),
        ("Number of RED events", len(red_events), "events"),
        ("Average RED per cycle  r̄", round(red_r, 4), "s"),
        ("Saturation check", "OVERSATURATED (μ ≤ λ)" if queue_oversaturated else "OK (μ > λ)", "—"),
    ]

    def _queue_sheet(name, formula_text, result_header, result_unit, pick, extra_inputs=()):
        """One sheet per queue formula: inputs block + per-RED-event results."""
        drop_existing(name)
        ws = wb.create_sheet(name)
        title(ws, name + "  (Section 12)", 4)
        formula_note(ws, 2, 4, formula_text, h=45)
        inputs = list(_QUEUE_COMMON_INPUTS) + list(extra_inputs)
        for col, h in enumerate(["Input", "Value", "Unit"], 1):
            H(ws.cell(4, col)); ws.cell(4, col).value = h
        for i, (n_, v_, u_) in enumerate(inputs):
            rr = 5 + i; alt = i % 2 == 1
            D(ws.cell(rr, 1), alt, align='left'); ws.cell(rr, 1).value = n_
            D(ws.cell(rr, 2), alt); ws.cell(rr, 2).value = v_
            D(ws.cell(rr, 3), alt); ws.cell(rr, 3).value = u_
        hq = 5 + len(inputs) + 1
        for col, h in enumerate(["RED event #", "r — effective RED (s)",
                                 result_header, "Unit"], 1):
            H(ws.cell(hq, col)); ws.cell(hq, col).value = h
        for i, (k, r_, TQ_, QM_, QL_) in enumerate(queue_rows):
            rr = hq + 1 + i; alt = i % 2 == 1
            val = pick(TQ_, QM_, QL_)
            D(ws.cell(rr, 1), alt); ws.cell(rr, 1).value = k
            D(ws.cell(rr, 2), alt, num=NUM4); ws.cell(rr, 2).value = round(r_, 4)
            D(ws.cell(rr, 3), alt, bold=True, num=NUM4)
            ws.cell(rr, 3).value = round(val, 4) if val is not None else "N/A (μ ≤ λ)"
            D(ws.cell(rr, 4), alt); ws.cell(rr, 4).value = result_unit
        ra = hq + 1 + len(queue_rows)
        avg_val = pick(TQ_avg, QM_avg, QL_avg)
        ws.cell(ra, 1).value = "AVERAGE (r̄ per cycle)"
        ws.cell(ra, 2).value = round(red_r, 4)
        ws.cell(ra, 3).value = round(avg_val, 4) if avg_val is not None else "N/A (μ ≤ λ)"
        ws.cell(ra, 4).value = result_unit
        total_row_style(ws, ra, 4)
        if not queue_rows:
            formula_note(ws, ra + 2, 4,
                         "No RED event was detected in this window — per-event rows are empty.")
        for col, w in zip("ABCD", [44, 22, 30, 12]):
            ws.column_dimensions[col].width = w

    _queue_sheet(
        "Queue Duration Time",
        "Formula (1):  T_Q = μ·r / (μ − λ).   T_Q = queue duration time (s); "
        "λ = arrival rate (LVU/h, Speed Log-Arrival); μ = service rate (LVU/h, "
        "Speed Log-Service); r = effective RED period (s, Section 2). Defined "
        "only when μ > λ; otherwise the queue does not dissipate (oversaturated).",
        "T_Q = μ·r/(μ−λ)", "s",
        lambda TQ, QM, QL: TQ,
        extra_inputs=[("LVU factors (class: factor)",
                       ", ".join(f"{k}:{v:g}" for k, v in LVU_FACTOR.items()), "—")],
    )
    _queue_sheet(
        "Max Queue Length",
        "Formula (2):  Q_M = λ·r / 3600.   Q_M = maximum queue length (light "
        "vehicle units); λ = arrival rate (LVU/h, Speed Log-Arrival); r = "
        "effective RED period (s); ÷3600 converts hours → seconds.",
        "Q_M = λ·r/3600", "LVU",
        lambda TQ, QM, QL: QM,
    )
    _queue_sheet(
        "Vehicle Queue Length",
        f"Formula (3):  Q_L = Q_M × {QL_M_PER_LVU:g} / L_M.   Q_L = vehicle queue "
        f"length (m); Q_M from formula (2); L_M = width of road entrance = "
        f"{ROAD_ENTRANCE_WIDTH_M:g} m (HARD-CODED as ROAD_ENTRANCE_WIDTH_M — edit "
        "it at the top of this cell).",
        f"Q_L = Q_M×{QL_M_PER_LVU:g}/L_M", "m",
        lambda TQ, QM, QL: QL,
        extra_inputs=[("L_M — width of road entrance (hard-coded)", ROAD_ENTRANCE_WIDTH_M, "m"),
                      (f"Metres per LVU (constant)", QL_M_PER_LVU, "m/LVU")],
    )

    # ── reorder: originals first, then analysis sheets ──
    analysis_names = ["Traffic Parameters", "Delay Estimation",
                      "Queue Duration Time", "Max Queue Length", "Vehicle Queue Length",
                      "Travel Time Cost", "Fuel Cost", "Electricity Cost",
                      "Pollution Cost", "VOC", "Cost Summary", "Reference Factors",
                      "Intersection Estimate"]
    desired = [s for s in wb.sheetnames if s not in analysis_names] \
              + [s for s in analysis_names if s in wb.sheetnames]
    wb._sheets.sort(key=lambda ws: desired.index(ws.title))

    # ════════════════════════════════════════════════════════════════════════════
    #  SAVE — ONE FINAL EXCEL FILE ONLY:  "<video>_analysis.xlsx"
    #  The final workbook contains every original sheet (Summary, Speed
    #  Log-Arrival, Speed Log-Service) PLUS all the analysis sheets above, so the
    #  intermediate detection report is no longer needed and is deleted.
    # ════════════════════════════════════════════════════════════════════════════
    def _stem(path):
        """basename without extension — robust to Windows '\\' even on a POSIX host."""
        base = str(path).replace("\\", "/").rstrip("/").split("/")[-1]
        return os.path.splitext(base)[0]

    # (1) input-video stem
    try:
        _vid_base = _stem(VIDEO_PATH)                       # produced by the detection cell
    except NameError:
        # stand-alone run: strip the known suffixes from the source-workbook stem
        _vid_base = _stem(SRC_XLSX)
        for _suf in ("_report", "_annotated", "_analysis"):
            if _vid_base.endswith(_suf):
                _vid_base = _vid_base[: -len(_suf)]

    # (2) the ONE output file: "<video>_analysis.xlsx" (same dir as detection xlsx)
    _out_dir = os.path.dirname(SRC_XLSX)
    ANALYSIS_OUTPUT = os.path.join(_out_dir, f"{_vid_base}_analysis.xlsx") if _out_dir \
                      else f"{_vid_base}_analysis.xlsx"
    wb.save(ANALYSIS_OUTPUT)
    print(f"\n✅ Final analysis workbook saved → {ANALYSIS_OUTPUT}")
    print(f"   Sheets: {wb.sheetnames}")

    # (3) remove the intermediate detection report so only ONE Excel file remains
    if DELETE_SOURCE_REPORT:
        try:
            if os.path.abspath(SRC_XLSX) != os.path.abspath(ANALYSIS_OUTPUT) and os.path.exists(SRC_XLSX):
                os.remove(SRC_XLSX)
                print(f"🗑️  Intermediate report deleted → {SRC_XLSX}")
                print("   (all of its sheets are preserved inside the final analysis workbook)")
        except OSError as _e:
            print(f"⚠️  Could not delete the intermediate report ({_e}) — delete it manually if unwanted.")

    # (4) verify the output filename matches the input video name
    print("\n────────── OUTPUT-NAMING CHECK ──────────")
    _expected_xlsx = _vid_base + "_analysis.xlsx"
    _actual_xlsx   = _stem(ANALYSIS_OUTPUT) + ".xlsx"
    print(f"Input video stem : {_vid_base}")
    print(f"  {'✅' if _actual_xlsx == _expected_xlsx else '❌'} Analysis Excel : {_actual_xlsx}"
          + ("" if _actual_xlsx == _expected_xlsx else f"   (expected {_expected_xlsx})"))
    if VIDEO_OUTPUT:
        _vid_ok = _stem(VIDEO_OUTPUT) == _vid_base + "_annotated"
        print(f"  {'✅' if _vid_ok else '❌'} Annotated video : {_stem(VIDEO_OUTPUT)}.mp4")
    print("✅ Only ONE Excel file is produced: the final analysis workbook above.")


    return ANALYSIS_OUTPUT


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  BATCH DRIVER — run this cell to process ALL videos in VIDEO_PATHS
#  ----------------------------------------------------------------------------
#  For EACH video it:
#    1) process_video()  → detection + tracking → "<video>_report.xlsx"
#                          + "<video>_annotated.mp4"
#    2) run_analysis()   → final "<video>_analysis.xlsx" (report sheets merged
#                          in, intermediate report deleted)
#  ⇒ 12 videos in VIDEO_PATHS → 12 Excel files, one single run.
#  A failed video is logged and skipped; the batch continues with the next one.
# ════════════════════════════════════════════════════════════════════════════
import os, time, traceback

batch_results = []
_batch_t0 = time.time()

for _i, _vp in enumerate(VIDEO_PATHS, 1):
    print("\n" + "═" * 78)
    print(f"🎬  VIDEO {_i}/{len(VIDEO_PATHS)} : {os.path.basename(_vp)}")
    print("═" * 78)
    if not os.path.exists(_vp):
        print(f"❌ File not found — skipping: {_vp}")
        batch_results.append((os.path.basename(_vp), "NOT FOUND", "—", 0.0))
        continue
    _t0 = time.time()
    try:
        _report_xlsx, _video_out = process_video(_vp)
        _final_xlsx = run_analysis(_report_xlsx, _vp, _video_out)
        batch_results.append((os.path.basename(_vp), "✅ OK", _final_xlsx,
                              time.time() - _t0))
    except KeyboardInterrupt:
        print("\n🛑 Batch stopped by user.")
        batch_results.append((os.path.basename(_vp), "🛑 INTERRUPTED", "—",
                              time.time() - _t0))
        break
    except Exception as _e:
        traceback.print_exc()
        print(f"❌ Failed on {os.path.basename(_vp)} : {_e} — continuing with next video.")
        batch_results.append((os.path.basename(_vp), "❌ FAILED", str(_e)[:60],
                              time.time() - _t0))
        continue

# ── Final batch summary ──
print("\n" + "═" * 78)
print(f"📋 BATCH SUMMARY — {len(batch_results)} video(s), "
      f"{(time.time() - _batch_t0) / 60:.1f} min total")
print("═" * 78)
print(f"{'#':>3}  {'Video':35s} {'Status':14s} {'Time':>8s}  Output")
for _j, (_name, _st, _out, _dt) in enumerate(batch_results, 1):
    print(f"{_j:>3}  {_name[:35]:35s} {_st:14s} {_dt/60:7.1f}m  {_out}")
_ok = sum(1 for r in batch_results if r[1] == "✅ OK")
print(f"\n✅ {_ok}/{len(VIDEO_PATHS)} Excel files written to: {OUTPUT_DIR}")
